# 1. Feature Engineering Environment and Validated Target Access

### Objective
This section initializes the feature-engineering environment and establishes access to the validated six-month early-warning target.

### Why this is necessary
The target construction has been completed and independently validated. Feature engineering must now be performed against this corrected target rather than the earlier target artifact.

The feature-engineering pipeline will combine:

1. The validated target dataset;
2. Loan-level characteristics available at the observation month;
3. Backward-looking payment behaviour;
4. Loan seasoning and balance-related variables;
5. Borrower and property characteristics.

### Modelling principle

All predictors must represent information available at or before the observation month `t`.

Future information must not enter the predictor set.

In particular:

- Current delinquency status will not be used as a predictor.
- Future delinquency information will never be used as a feature.
- Target variables will remain separate from predictors.
- Variables containing post-event information will be excluded.
- Payment history will be transformed into backward-looking behavioural features.

### Data source

The validated target is:

`fannie_2018Q1_early_warning_target_corrected.parquet`

The original processed Fannie Mae loan-performance chunks remain the source for predictor variables.

No feature transformation is performed in this cell.

In [1]:
# ================================================================
# FEATURE ENGINEERING ENVIRONMENT
# ================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import duckdb
import glob
import gc

# ---------------------------------------------------------------
# Project paths
# ---------------------------------------------------------------

PROJECT_DIR = Path(
    r"C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning"
)

DATA_DIR = PROJECT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"

CHUNKS_DIR = (
    PROCESSED_DIR /
    "fannie_2018Q1_chunks"
)

TARGET_FILE = (
    PROCESSED_DIR /
    "fannie_2018Q1_early_warning_target_corrected.parquet"
)

FEATURE_OUTPUT_DIR = (
    PROCESSED_DIR /
    "feature_engineering"
)

FEATURE_TEMP_DIR = (
    FEATURE_OUTPUT_DIR /
    "duckdb_temp"
)

FEATURE_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FEATURE_TEMP_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RAW_GLOB = str(
    CHUNKS_DIR /
    "part_*.csv.gz"
)

FILES = sorted(
    glob.glob(RAW_GLOB)
)

# ---------------------------------------------------------------
# Basic path validation
# ---------------------------------------------------------------

print("=" * 70)
print("FEATURE ENGINEERING ENVIRONMENT")
print("=" * 70)

print(f"Project directory : {PROJECT_DIR}")
print(f"Chunks directory  : {CHUNKS_DIR}")
print(f"Target file       : {TARGET_FILE}")
print(f"Number of chunks  : {len(FILES)}")

print("\nPath checks:")
print(f"Project exists    : {PROJECT_DIR.exists()}")
print(f"Chunks exist      : {CHUNKS_DIR.exists()}")
print(f"Target exists     : {TARGET_FILE.exists()}")

if not PROJECT_DIR.exists():
    raise FileNotFoundError(
        f"Project directory not found: {PROJECT_DIR}"
    )

if not CHUNKS_DIR.exists():
    raise FileNotFoundError(
        f"Chunks directory not found: {CHUNKS_DIR}"
    )

if len(FILES) != 95:
    raise ValueError(
        f"Expected 95 processed chunks, found {len(FILES)}."
    )

if not TARGET_FILE.exists():
    raise FileNotFoundError(
        "Validated target file was not found."
    )

# ---------------------------------------------------------------
# Fresh DuckDB connection
# ---------------------------------------------------------------

feature_con = duckdb.connect()

# Configure BEFORE executing data queries.
feature_con.execute(
    "SET memory_limit = '4GB'"
)

feature_con.execute(
    "SET threads = 2"
)

feature_con.execute(
    "SET preserve_insertion_order = false"
)

feature_con.execute(
    f"SET temp_directory = '{FEATURE_TEMP_DIR}'"
)

print("\nDuckDB:")
print(f"Version           : {duckdb.__version__}")
print("Memory limit      : 4 GB")
print("Threads           : 2")
print(f"Temporary folder  : {FEATURE_TEMP_DIR}")

FEATURE ENGINEERING ENVIRONMENT
Project directory : C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning
Chunks directory  : C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning\data\processed\fannie_2018Q1_chunks
Target file       : C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning\data\processed\fannie_2018Q1_early_warning_target_corrected.parquet
Number of chunks  : 95

Path checks:
Project exists    : True
Chunks exist      : True
Target exists     : True

DuckDB:
Version           : 1.5.5
Memory limit      : 4 GB
Threads           : 2
Temporary folder  : C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning\data\processed\feature_engineering\duckdb_temp


# 2. Validate the Corrected Early-Warning Target

### Objective
This section verifies that the permanently saved target dataset is identical to the validated target produced in the previous notebook.

### Why this is necessary
Feature engineering will use the saved target as the foundation of the modelling dataset. Before joining predictor variables, we must confirm that:

- the target file is readable;
- only valid target values (`0` and `1`) are present;
- the expected number of observations is preserved;
- the expected number of unique loans is preserved;
- the event and non-event counts are unchanged;
- the temporal coverage is correct.

This provides a reproducible checkpoint between target construction and feature engineering.

### Expected values

- Valid observations: 20,155,269
- Unique loans: 450,106
- Events: 646,957
- Non-events: 19,508,312
- Earliest observation: January 2018
- Latest observation: September 2025

No source data is modified in this step.

In [2]:
# ================================================================
# VALIDATE SAVED EARLY-WARNING TARGET
# ================================================================

target_check = feature_con.execute(f"""
SELECT
    COUNT(*) AS observations,

    COUNT(DISTINCT loan_id) AS unique_loans,

    COUNT(*) FILTER (
        WHERE target_6m = 1
    ) AS events,

    COUNT(*) FILTER (
        WHERE target_6m = 0
    ) AS non_events,

    COUNT(*) FILTER (
        WHERE target_6m IS NULL
    ) AS null_targets,

    COUNT(DISTINCT target_6m) AS distinct_target_values,

    MIN(reporting_date) AS earliest_date,

    MAX(reporting_date) AS latest_date

FROM read_parquet(
    '{TARGET_FILE}'
)
""").fetchdf()

print("=" * 70)
print("SAVED TARGET VALIDATION")
print("=" * 70)

display(target_check)

# ---------------------------------------------------------------
# Automated validation checks
# ---------------------------------------------------------------

row_count = int(
    target_check.loc[0, "observations"]
)

unique_loans = int(
    target_check.loc[0, "unique_loans"]
)

events = int(
    target_check.loc[0, "events"]
)

non_events = int(
    target_check.loc[0, "non_events"]
)

null_targets = int(
    target_check.loc[0, "null_targets"]
)

distinct_targets = int(
    target_check.loc[0, "distinct_target_values"]
)

earliest_date = str(
    target_check.loc[0, "earliest_date"]
)

latest_date = str(
    target_check.loc[0, "latest_date"]
)

checks = {
    "Observation count": row_count == 20_155_269,
    "Unique loan count": unique_loans == 450_106,
    "Event count": events == 646_957,
    "Non-event count": non_events == 19_508_312,
    "No NULL targets": null_targets == 0,
    "Only two target classes": distinct_targets == 2,
    "Earliest date": earliest_date.startswith("2018-01-01"),
    "Latest date": latest_date.startswith("2025-09-01")
}

print("\n" + "=" * 70)
print("VALIDATION CHECKS")
print("=" * 70)

for check_name, passed in checks.items():
    print(
        f"{check_name:<30}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

if not all(checks.values()):
    raise ValueError(
        "One or more target validation checks failed."
    )

print("\nSTATUS: PASS — Saved target matches the validated target.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

SAVED TARGET VALIDATION


,observations,unique_loans,events,non_events,null_targets,distinct_target_values,earliest_date,latest_date
0,20155269,450106,646957,19508312,0,2,2018-01-01,2025-09-01



VALIDATION CHECKS
Observation count             : PASS
Unique loan count             : PASS
Event count                   : PASS
Non-event count               : PASS
No NULL targets               : PASS
Only two target classes       : PASS
Earliest date                 : PASS
Latest date                   : PASS

STATUS: PASS — Saved target matches the validated target.


# 3. Feature Design and Leakage Control

### Objective
This section defines the predictor variables that will be used to construct the modelling dataset.

### Why this is necessary
A credit-risk early-warning model must use information that would have been available at the prediction month `t`.

The feature design therefore separates source variables into:

1. **Direct predictors** — variables that can be used after appropriate cleaning or transformation.
2. **Behavioural predictors** — backward-looking features derived from payment history.
3. **Derived predictors** — variables calculated from existing source variables.
4. **Excluded variables** — identifiers, target-defining variables, future/post-event variables, or variables with unsuitable information content.

### Leakage-control rules

The following variables are excluded from the predictor matrix:

- `loan_id` — identifier only.
- `monthly_reporting_period` — retained as the temporal index but not used directly as a predictor.
- `current_delinquency_status` — defines target eligibility and therefore must not be used as a predictor.
- `zero_balance_code` — post-performance outcome information.
- `zero_balance_effective_date` — post-performance timing information.
- `mi_cancellation_indicator` — may reflect subsequent loan performance.
- `interest_only_first_pi_date` — largely non-informative for the modelling population and structurally unavailable.
- `months_to_amortization` — structurally unavailable in this dataset.
- `loan_payment_history` — not used as a raw categorical variable; it will be transformed into backward-looking behavioural features.

### Conditionally applicable variables

The following variables contain legitimate structural missingness and will not be blindly imputed:

- `co_borrower_credit_score`
- `mortgage_insurance_percentage`

Their missingness will be represented appropriately during feature engineering.

### Temporal principle

Every engineered feature must be computable using information available at or before month `t`.

The resulting feature dataset will therefore contain:

- loan identifier;
- observation month;
- engineered predictors;
- validated six-month target.

The target remains completely separate from feature construction.

In [3]:
# ================================================================
# FEATURE DESIGN SPECIFICATION
# ================================================================

# ---------------------------------------------------------------
# Identifiers / temporal fields
# ---------------------------------------------------------------

IDENTIFIER_COLUMNS = [
    "loan_id"
]

TIME_COLUMNS = [
    "monthly_reporting_period"
]

# ---------------------------------------------------------------
# Target / eligibility variables
# ---------------------------------------------------------------

TARGET_COLUMNS = [
    "current_delinquency_status"
]

# ---------------------------------------------------------------
# Behavioural source
# ---------------------------------------------------------------

BEHAVIOURAL_COLUMNS = [
    "loan_payment_history"
]

# ---------------------------------------------------------------
# Variables excluded because they contain post-event information,
# structural non-applicability, or unsuitable modelling content.
# ---------------------------------------------------------------

EXCLUDED_COLUMNS = [
    "interest_only_first_pi_date",
    "months_to_amortization",
    "mi_cancellation_indicator",
    "zero_balance_code",
    "zero_balance_effective_date"
]

# ---------------------------------------------------------------
# Conditionally applicable predictors
# ---------------------------------------------------------------

CONDITIONAL_COLUMNS = [
    "co_borrower_credit_score",
    "mortgage_insurance_percentage"
]

# ---------------------------------------------------------------
# Candidate direct predictors
# ---------------------------------------------------------------

DIRECT_PREDICTOR_COLUMNS = [
    "channel",
    "seller_name",
    "servicer_name",
    "original_interest_rate",
    "current_interest_rate",
    "original_upb",
    "current_actual_upb",
    "original_loan_term",
    "origination_date",
    "first_payment_date",
    "loan_age",
    "remaining_months_legal_maturity",
    "remaining_months_maturity",
    "maturity_date",
    "original_ltv",
    "original_cltv",
    "number_of_borrowers",
    "dti",
    "borrower_credit_score",
    "first_time_home_buyer",
    "loan_purpose",
    "property_type",
    "number_of_units",
    "occupancy_status",
    "property_state",
    "msa",
    "zip_code_short",
    "amortization_type",
    "prepayment_penalty_indicator",
    "interest_only_indicator",
    "modification_flag"
]

# ---------------------------------------------------------------
# Combined predictor specification
# ---------------------------------------------------------------

ALL_PREDICTOR_SOURCE_COLUMNS = (
    DIRECT_PREDICTOR_COLUMNS
    + CONDITIONAL_COLUMNS
    + BEHAVIOURAL_COLUMNS
)

# Remove accidental duplicates while preserving order.
ALL_PREDICTOR_SOURCE_COLUMNS = list(
    dict.fromkeys(
        ALL_PREDICTOR_SOURCE_COLUMNS
    )
)

# ---------------------------------------------------------------
# Display the specification
# ---------------------------------------------------------------

feature_design = []

for col in DIRECT_PREDICTOR_COLUMNS:
    feature_design.append({
        "source_column": col,
        "role": "DIRECT_PREDICTOR"
    })

for col in CONDITIONAL_COLUMNS:
    feature_design.append({
        "source_column": col,
        "role": "CONDITIONAL_PREDICTOR"
    })

for col in BEHAVIOURAL_COLUMNS:
    feature_design.append({
        "source_column": col,
        "role": "BEHAVIOURAL_SOURCE"
    })

for col in IDENTIFIER_COLUMNS:
    feature_design.append({
        "source_column": col,
        "role": "IDENTIFIER"
    })

for col in TIME_COLUMNS:
    feature_design.append({
        "source_column": col,
        "role": "TIME_INDEX"
    })

for col in TARGET_COLUMNS:
    feature_design.append({
        "source_column": col,
        "role": "TARGET / ELIGIBILITY"
    })

for col in EXCLUDED_COLUMNS:
    feature_design.append({
        "source_column": col,
        "role": "EXCLUDED"
    })

feature_design_df = pd.DataFrame(
    feature_design
)

print("=" * 70)
print("FEATURE DESIGN SPECIFICATION")
print("=" * 70)

display(feature_design_df)

print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)

print(
    f"Direct predictors       : "
    f"{len(DIRECT_PREDICTOR_COLUMNS)}"
)

print(
    f"Conditional predictors  : "
    f"{len(CONDITIONAL_COLUMNS)}"
)

print(
    f"Behavioural sources     : "
    f"{len(BEHAVIOURAL_COLUMNS)}"
)

print(
    f"Excluded variables      : "
    f"{len(EXCLUDED_COLUMNS)}"
)

print(
    f"Total source predictors : "
    f"{len(ALL_PREDICTOR_SOURCE_COLUMNS)}"
)

FEATURE DESIGN SPECIFICATION


,source_column,role
0,channel,DIRECT_PREDICTOR
1,seller_name,DIRECT_PREDICTOR
2,servicer_name,DIRECT_PREDICTOR
3,original_interest_rate,DIRECT_PREDICTOR
4,current_interest_rate,DIRECT_PREDICTOR
5,original_upb,DIRECT_PREDICTOR
6,current_actual_upb,DIRECT_PREDICTOR
7,original_loan_term,DIRECT_PREDICTOR
8,origination_date,DIRECT_PREDICTOR
9,first_payment_date,DIRECT_PREDICTOR



SUMMARY
Direct predictors       : 31
Conditional predictors  : 2
Behavioural sources     : 1
Excluded variables      : 5
Total source predictors : 34


# 4. Validate Target-to-Raw Observation Coverage

### Objective
This section verifies that every observation in the validated target dataset has a corresponding loan-month record in the original processed Fannie Mae performance data.

### Why this is necessary
The modelling dataset will be created by joining the validated target to predictor variables from the raw performance records.

Before performing this join, we must establish that:

- every target loan-month exists in the source data;
- no target observation maps to multiple source records;
- the target observation dates are within the source reporting period;
- the target-to-source relationship is one-to-one.

### Why this matters

A missing or duplicated source record could cause:

- loss of modelling observations;
- duplicated training observations;
- incorrect feature values;
- biased model estimates.

The previous loan-month uniqueness validation established that the source panel itself is unique. This step independently verifies the relationship between the final target dataset and the source panel.

Only identifiers and reporting dates are compared in this validation. No predictor variables are transformed or modelled.

In [4]:
# ================================================================
# TARGET-TO-RAW OBSERVATION COVERAGE VALIDATION
# ================================================================

# We use a fresh connection for this large scan so that the
# feature-engineering connection remains clean.

coverage_con = duckdb.connect()

coverage_temp_dir = (
    FEATURE_OUTPUT_DIR /
    "coverage_temp"
)

coverage_temp_dir.mkdir(
    parents=True,
    exist_ok=True
)

coverage_con.execute(
    "SET memory_limit = '4GB'"
)

coverage_con.execute(
    "SET threads = 2"
)

coverage_con.execute(
    "SET preserve_insertion_order = false"
)

coverage_con.execute(
    f"SET temp_directory = '{coverage_temp_dir}'"
)

coverage_query = f"""
WITH target AS (

    SELECT
        loan_id,
        reporting_date
    FROM read_parquet(
        '{TARGET_FILE}'
    )

),

source_panel AS (

    SELECT
        loan_id,

        STRPTIME(
            monthly_reporting_period,
            '%m%Y'
        ) AS reporting_date

    FROM read_csv_auto(
        '{RAW_GLOB}',
        header = true,
        compression = 'gzip',
        union_by_name = true
    )

),

matched AS (

    SELECT
        t.loan_id,
        t.reporting_date,
        COUNT(s.loan_id) AS source_matches
    FROM target t
    LEFT JOIN source_panel s
        ON t.loan_id = s.loan_id
       AND t.reporting_date = s.reporting_date
    GROUP BY
        t.loan_id,
        t.reporting_date
)

SELECT
    COUNT(*) AS target_observations,

    COUNT(*) FILTER (
        WHERE source_matches = 0
    ) AS target_observations_without_source,

    COUNT(*) FILTER (
        WHERE source_matches = 1
    ) AS target_observations_with_one_source,

    COUNT(*) FILTER (
        WHERE source_matches > 1
    ) AS target_observations_with_multiple_sources,

    MIN(reporting_date) AS earliest_target_date,

    MAX(reporting_date) AS latest_target_date

FROM matched
"""

coverage_df = coverage_con.execute(
    coverage_query
).fetchdf()

print("=" * 70)
print("TARGET-TO-RAW OBSERVATION COVERAGE")
print("=" * 70)

display(coverage_df)

# ---------------------------------------------------------------
# Automated validation
# ---------------------------------------------------------------

without_source = int(
    coverage_df.loc[
        0,
        "target_observations_without_source"
    ]
)

multiple_sources = int(
    coverage_df.loc[
        0,
        "target_observations_with_multiple_sources"
    ]
)

target_observations = int(
    coverage_df.loc[
        0,
        "target_observations"
    ]
)

checks = {
    "Target observations": target_observations == 20_155_269,
    "No missing source records": without_source == 0,
    "No duplicate source matches": multiple_sources == 0
}

print("\n" + "=" * 70)
print("COVERAGE VALIDATION CHECKS")
print("=" * 70)

for check_name, passed in checks.items():
    print(
        f"{check_name:<35}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

if not all(checks.values()):
    raise ValueError(
        "Target-to-source coverage validation failed."
    )

print(
    "\nSTATUS: PASS — Every target observation "
    "maps to exactly one source loan-month."
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

TARGET-TO-RAW OBSERVATION COVERAGE


,target_observations,target_observations_without_source,target_observations_with_one_source,target_observations_with_multiple_sources,earliest_target_date,latest_target_date
0,20155269,0,20155269,0,2018-01-01,2025-09-01



COVERAGE VALIDATION CHECKS
Target observations                : PASS
No missing source records          : PASS
No duplicate source matches        : PASS

STATUS: PASS — Every target observation maps to exactly one source loan-month.


# 5. Construct the Base Modelling Dataset

### Objective
This section creates the base modelling dataset by joining the validated six-month early-warning target with predictor variables observed at the same loan-month.

### Why this is necessary
The validated target contains only:

- loan identifier;
- observation month;
- six-month outcome.

The predictor variables must therefore be retrieved from the original loan-performance records and joined to the target at the exact same loan-month.

### Construction principle

For every target observation at month `t`, only information recorded at month `t` is extracted.

The current delinquency status is deliberately excluded from the predictor dataset because it was used to define the eligible population and therefore could introduce target-related information into the model.

The raw payment-history field is retained temporarily because it will be transformed into explicitly backward-looking behavioural features in the next stage.

### Efficiency consideration

The source contains more than 23 million observations. DuckDB is used to perform the join and write the resulting dataset directly to Parquet rather than loading the full dataset into pandas.

Only the observations that have a valid target (`target_6m = 0 or 1`) are retained.

No statistical imputation or categorical encoding is performed in this step.

In [5]:
# ================================================================
# CONSTRUCT BASE MODELLING DATASET
# ================================================================

BASE_FEATURE_FILE = (
    FEATURE_OUTPUT_DIR /
    "fannie_2018Q1_base_modelling_dataset.parquet"
)

# ---------------------------------------------------------------
# Remove previous output if this cell is intentionally rerun.
# ---------------------------------------------------------------

if BASE_FEATURE_FILE.exists():
    BASE_FEATURE_FILE.unlink()

print("=" * 70)
print("CONSTRUCTING BASE MODELLING DATASET")
print("=" * 70)

print("Target file:")
print(TARGET_FILE)

print("\nSource chunks:")
print(len(FILES))

print("\nWriting:")
print(BASE_FEATURE_FILE)

# ---------------------------------------------------------------
# Select only variables required at this stage.
#
# IMPORTANT:
# current_delinquency_status is intentionally excluded.
# loan_payment_history is retained for later transformation.
# ---------------------------------------------------------------

base_query = f"""
COPY (

    SELECT

        /* Identification / temporal index */
        t.loan_id,
        t.reporting_date,

        /* Target */
        t.target_6m,

        /* Direct predictors */
        r.channel,
        r.seller_name,
        r.servicer_name,

        r.original_interest_rate,
        r.current_interest_rate,

        r.original_upb,
        r.current_actual_upb,

        r.original_loan_term,

        r.origination_date,
        r.first_payment_date,

        r.loan_age,

        r.remaining_months_legal_maturity,
        r.remaining_months_maturity,

        r.maturity_date,

        r.original_ltv,
        r.original_cltv,

        r.number_of_borrowers,
        r.dti,

        r.borrower_credit_score,
        r.co_borrower_credit_score,

        r.first_time_home_buyer,
        r.loan_purpose,
        r.property_type,

        r.number_of_units,

        r.occupancy_status,
        r.property_state,
        r.msa,
        r.zip_code_short,

        r.mortgage_insurance_percentage,

        r.amortization_type,
        r.prepayment_penalty_indicator,
        r.interest_only_indicator,

        r.modification_flag,

        /* Behavioural source retained temporarily */
        r.loan_payment_history

    FROM read_parquet(
        '{TARGET_FILE}'
    ) t

    INNER JOIN read_csv_auto(
        '{RAW_GLOB}',
        header = true,
        compression = 'gzip',
        union_by_name = true
    ) r

        ON t.loan_id = r.loan_id

       AND t.reporting_date =
           STRPTIME(
               r.monthly_reporting_period,
               '%m%Y'
           )

)
TO '{BASE_FEATURE_FILE}'
(
    FORMAT PARQUET,
    COMPRESSION ZSTD
)
"""

feature_con.execute(base_query)

print("\n" + "=" * 70)
print("BASE DATASET CONSTRUCTION COMPLETED")
print("=" * 70)

print(f"Output file: {BASE_FEATURE_FILE}")

# ---------------------------------------------------------------
# Verify the resulting dataset.
# ---------------------------------------------------------------

base_check = feature_con.execute(f"""
SELECT
    COUNT(*) AS observations,
    COUNT(DISTINCT loan_id) AS unique_loans,
    COUNT(DISTINCT reporting_date) AS reporting_months,
    COUNT(*) FILTER (
        WHERE target_6m = 1
    ) AS events,
    COUNT(*) FILTER (
        WHERE target_6m = 0
    ) AS non_events,
    MIN(reporting_date) AS earliest_date,
    MAX(reporting_date) AS latest_date
FROM read_parquet(
    '{BASE_FEATURE_FILE}'
)
""").fetchdf()

display(base_check)

print("\nFile size (MB):")
print(
    round(
        BASE_FEATURE_FILE.stat().st_size /
        (1024 ** 2),
        2
    )
)

CONSTRUCTING BASE MODELLING DATASET
Target file:
C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning\data\processed\fannie_2018Q1_early_warning_target_corrected.parquet

Source chunks:
95

Writing:
C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning\data\processed\feature_engineering\fannie_2018Q1_base_modelling_dataset.parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


BASE DATASET CONSTRUCTION COMPLETED
Output file: C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning\data\processed\feature_engineering\fannie_2018Q1_base_modelling_dataset.parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,observations,unique_loans,reporting_months,events,non_events,earliest_date,latest_date
0,20155269,450106,93,646957,19508312,2018-01-01,2025-09-01



File size (MB):
448.55


# 6. Base Dataset Schema and Leakage Check

### Objective
This section verifies the structure of the base modelling dataset created in the previous step.

### Why this is necessary
Before engineering derived features, we need to confirm that the base dataset contains exactly the intended modelling inputs.

In particular, this check verifies that:

- the loan identifier is present;
- the observation date is present;
- the validated target is present;
- the raw behavioural source is present for later transformation;
- the current delinquency status is NOT present as a predictor;
- excluded post-event variables have not been included.

This is an explicit leakage-control checkpoint.

No data is modified in this step.

In [6]:
# ================================================================
# BASE DATASET SCHEMA AND LEAKAGE CHECK
# ================================================================

base_schema_df = feature_con.execute(f"""
DESCRIBE
SELECT *
FROM read_parquet(
    '{BASE_FEATURE_FILE}'
)
""").fetchdf()

print("=" * 70)
print("BASE MODELLING DATASET SCHEMA")
print("=" * 70)

print(
    f"Number of columns: {len(base_schema_df)}"
)

display(base_schema_df)

# ---------------------------------------------------------------
# Extract column names
# ---------------------------------------------------------------

base_columns = base_schema_df["column_name"].tolist()

# ---------------------------------------------------------------
# Explicit leakage checks
# ---------------------------------------------------------------

leakage_variables = [
    "current_delinquency_status",
    "zero_balance_code",
    "zero_balance_effective_date",
    "mi_cancellation_indicator"
]

unexpected_leakage_columns = [
    col for col in leakage_variables
    if col in base_columns
]

required_columns = [
    "loan_id",
    "reporting_date",
    "target_6m",
    "loan_payment_history"
]

missing_required_columns = [
    col for col in required_columns
    if col not in base_columns
]

print("\n" + "=" * 70)
print("LEAKAGE AND STRUCTURE CHECK")
print("=" * 70)

print(
    "Unexpected leakage variables present:",
    unexpected_leakage_columns
)

print(
    "Required columns missing:",
    missing_required_columns
)

if unexpected_leakage_columns:
    raise ValueError(
        "Potential leakage variables detected in the base dataset."
    )

if missing_required_columns:
    raise ValueError(
        "Required base modelling columns are missing."
    )

print(
    "\nSTATUS: PASS — Base dataset contains the intended "
    "modelling inputs and excludes identified leakage variables."
)

BASE MODELLING DATASET SCHEMA
Number of columns: 37


,column_name,column_type,null,key,default,extra
0,loan_id,BIGINT,YES,None,None,None
1,reporting_date,TIMESTAMP,YES,None,None,None
2,target_6m,INTEGER,YES,None,None,None
3,channel,VARCHAR,YES,None,None,None
4,seller_name,VARCHAR,YES,None,None,None
5,servicer_name,VARCHAR,YES,None,None,None
6,original_interest_rate,DOUBLE,YES,None,None,None
7,current_interest_rate,DOUBLE,YES,None,None,None
8,original_upb,DOUBLE,YES,None,None,None
9,current_actual_upb,DOUBLE,YES,None,None,None



LEAKAGE AND STRUCTURE CHECK
Unexpected leakage variables present: []
Required columns missing: []

STATUS: PASS — Base dataset contains the intended modelling inputs and excludes identified leakage variables.


# 7. Payment History Parsing Test

### Objective
This section tests the interpretation of the Fannie Mae `loan_payment_history` field on a small sample before applying the transformation to the full modelling dataset.

### Why this is necessary
Payment history is stored as a compact character string in which every two characters represent one monthly delinquency status.

The rightmost two-character code represents the most recent month relative to the observation.

For the early-warning model, the current observation month `t` must not be included in the behavioural predictor window. Therefore, the rightmost code is treated as the current-month status and excluded.

The remaining codes represent prior monthly behaviour.

### Features to be created

Using the 24 months preceding `t`:

- prior 3-month delinquency indicator;
- prior 6-month delinquency indicator;
- prior 12-month delinquency indicator;
- prior 24-month delinquency indicator;
- maximum prior delinquency;
- delinquency count over 6 months;
- delinquency count over 12 months;
- delinquency count over 24 months.

`XX` codes are treated as unknown rather than as delinquency.

This cell only tests the parsing logic on a small sample. No full-dataset transformation is performed.

In [7]:
# ================================================================
# PAYMENT HISTORY PARSING TEST
# ================================================================

# ---------------------------------------------------------------
# Retrieve a small sample of non-missing payment histories.
# ---------------------------------------------------------------

payment_test_df = feature_con.execute(f"""
SELECT
    loan_id,
    reporting_date,
    loan_payment_history
FROM read_parquet(
    '{BASE_FEATURE_FILE}'
)
WHERE loan_payment_history IS NOT NULL
  AND TRIM(loan_payment_history) <> ''
LIMIT 10
""").fetchdf()

# ---------------------------------------------------------------
# Parsing function
# ---------------------------------------------------------------

def parse_payment_history(history):

    if pd.isna(history) or str(history).strip() == "":
        return {
            "ph_missing": 1,
            "ph_prior_3m_delinquency": np.nan,
            "ph_prior_6m_delinquency": np.nan,
            "ph_prior_12m_delinquency": np.nan,
            "ph_prior_24m_delinquency": np.nan,
            "ph_prior_max_delinquency": np.nan,
            "ph_prior_delinquency_count_6m": np.nan,
            "ph_prior_delinquency_count_12m": np.nan,
            "ph_prior_delinquency_count_24m": np.nan
        }

    history = str(history).strip()

    # Each monthly status consists of two characters.
    monthly_codes = [
        history[i:i+2]
        for i in range(0, len(history), 2)
    ]

    # The rightmost code represents the current month.
    # Therefore, exclude it from backward-looking features.
    prior_codes = monthly_codes[:-1]

    # Keep only the previous 24 months.
    prior_codes = prior_codes[-24:]

    def numeric_values(codes):
        values = []

        for code in codes:
            if code.isdigit():
                values.append(int(code))

        return values

    prior_3 = prior_codes[-3:]
    prior_6 = prior_codes[-6:]
    prior_12 = prior_codes[-12:]
    prior_24 = prior_codes[-24:]

    values_3 = numeric_values(prior_3)
    values_6 = numeric_values(prior_6)
    values_12 = numeric_values(prior_12)
    values_24 = numeric_values(prior_24)

    return {
        "ph_missing": 0,

        "ph_prior_3m_delinquency":
            int(any(v >= 1 for v in values_3)),

        "ph_prior_6m_delinquency":
            int(any(v >= 1 for v in values_6)),

        "ph_prior_12m_delinquency":
            int(any(v >= 1 for v in values_12)),

        "ph_prior_24m_delinquency":
            int(any(v >= 1 for v in values_24)),

        "ph_prior_max_delinquency":
            max(values_24) if values_24 else np.nan,

        "ph_prior_delinquency_count_6m":
            sum(v >= 1 for v in values_6),

        "ph_prior_delinquency_count_12m":
            sum(v >= 1 for v in values_12),

        "ph_prior_delinquency_count_24m":
            sum(v >= 1 for v in values_24
        )
    }


# ---------------------------------------------------------------
# Apply parser to test sample
# ---------------------------------------------------------------

parsed_test = payment_test_df[
    "loan_payment_history"
].apply(parse_payment_history)

parsed_test_df = pd.concat(
    [
        payment_test_df.reset_index(drop=True),
        pd.DataFrame(parsed_test.tolist())
    ],
    axis=1
)

print("=" * 70)
print("PAYMENT HISTORY PARSING TEST")
print("=" * 70)

display(parsed_test_df)

PAYMENT HISTORY PARSING TEST


,loan_id,reporting_date,loan_payment_history,ph_missing,ph_prior_3m_delinquency,ph_prior_6m_delinquency,ph_prior_12m_delinquency,ph_prior_24m_delinquency,ph_prior_max_delinquency,ph_prior_delinquency_count_6m,ph_prior_delinquency_count_12m,ph_prior_delinquency_count_24m
0,835403805153,2020-06-01,000000000000000000000000000000000000000000000000,0,0,0,0,0,0,0,0,0
1,835410386949,2020-06-01,000000000000000000000000000000000000000000000000,0,0,0,0,0,0,0,0,0
2,835418278429,2020-06-01,000000000000000000000000000000000000000000000000,0,0,0,0,0,0,0,0,0
3,835431738079,2020-06-01,000000010000000000000000000000000000000000000000,0,0,0,0,1,1,0,0,1
4,835434167764,2020-06-01,000000000000000000000000000000000000000000000000,0,0,0,0,0,0,0,0,0
5,835437648684,2020-06-01,000000000000000000000000000000000000000000000000,0,0,0,0,0,0,0,0,0
6,835472578643,2020-06-01,000000000000000000000000000000000000000000000000,0,0,0,0,0,0,0,0,0
7,835475726852,2020-06-01,000000000000000000000000000000000000000000000000,0,0,0,0,0,0,0,0,0
8,835499979283,2020-06-01,000000000000000000000000000000000000000000000000,0,0,0,0,0,0,0,0,0
9,835544366147,2020-06-01,000000000000000000000000000000000000000000000000,0,0,0,0,0,0,0,0,0


# 8. Payment History Structural Validation

### Objective
This section validates the structural consistency of the payment-history field before creating behavioural predictors for the full modelling dataset.

### Why this is necessary
The payment-history transformation assumes that:

- each monthly status occupies two characters;
- the rightmost two-character code represents the current month;
- earlier codes represent historical monthly behaviour;
- the current-month code can therefore be excluded from backward-looking features.

We verify these assumptions empirically by examining:

1. payment-history string lengths;
2. whether lengths are compatible with two-character monthly coding;
3. the frequency of the rightmost status code;
4. agreement between the rightmost payment-history code and `current_delinquency_status`.

The current delinquency status is available in the raw source but is **used only for this validation**. It will not be included as a model predictor.

No feature dataset is created in this step.

In [8]:
# ================================================================
# PAYMENT HISTORY STRUCTURAL VALIDATION
# ================================================================

print("=" * 70)
print("PAYMENT HISTORY LENGTH DISTRIBUTION")
print("=" * 70)

length_query = f"""
SELECT
    LENGTH(loan_payment_history) AS history_length,
    COUNT(*) AS observations
FROM read_parquet(
    '{BASE_FEATURE_FILE}'
)
WHERE loan_payment_history IS NOT NULL
GROUP BY
    history_length
ORDER BY
    history_length
"""

payment_length_df = feature_con.execute(
    length_query
).fetchdf()

display(payment_length_df)

# ---------------------------------------------------------------
# Validate whether all observed lengths are even.
# ---------------------------------------------------------------

odd_length_count = feature_con.execute(f"""
SELECT
    COUNT(*) AS odd_length_observations
FROM read_parquet(
    '{BASE_FEATURE_FILE}'
)
WHERE loan_payment_history IS NOT NULL
  AND MOD(
      LENGTH(loan_payment_history),
      2
  ) <> 0
""").fetchone()[0]

print(
    f"Observations with odd history length: "
    f"{odd_length_count:,}"
)

# ---------------------------------------------------------------
# Rightmost payment-history code.
# ---------------------------------------------------------------

print("\n" + "=" * 70)
print("RIGHTMOST PAYMENT-HISTORY CODE")
print("=" * 70)

rightmost_query = f"""
SELECT
    RIGHT(
        loan_payment_history,
        2
    ) AS payment_history_current_code,

    current_delinquency_status,

    COUNT(*) AS observations

FROM read_csv_auto(
    '{RAW_GLOB}',
    header = true,
    compression = 'gzip',
    union_by_name = true
)

WHERE loan_payment_history IS NOT NULL
  AND TRIM(loan_payment_history) <> ''

GROUP BY
    payment_history_current_code,
    current_delinquency_status

ORDER BY
    observations DESC

LIMIT 30
"""

rightmost_df = feature_con.execute(
    rightmost_query
).fetchdf()

display(rightmost_df)

# ---------------------------------------------------------------
# Exact agreement between rightmost payment-history code and
# current delinquency status.
#
# XX is also compared directly.
# ---------------------------------------------------------------

agreement_query = f"""
SELECT
    COUNT(*) AS observations_checked,

    COUNT(*) FILTER (
        WHERE RIGHT(
            loan_payment_history,
            2
        ) = current_delinquency_status
    ) AS exact_matches,

    COUNT(*) FILTER (
        WHERE RIGHT(
            loan_payment_history,
            2
        ) <> current_delinquency_status
    ) AS mismatches

FROM read_csv_auto(
    '{RAW_GLOB}',
    header = true,
    compression = 'gzip',
    union_by_name = true
)

WHERE loan_payment_history IS NOT NULL
  AND TRIM(loan_payment_history) <> ''
"""

agreement_df = feature_con.execute(
    agreement_query
).fetchdf()

print("\n" + "=" * 70)
print("PAYMENT HISTORY / CURRENT STATUS AGREEMENT")
print("=" * 70)

display(agreement_df)

# ---------------------------------------------------------------
# Final checks
# ---------------------------------------------------------------

mismatches = int(
    agreement_df.loc[0, "mismatches"]
)

print("\n" + "=" * 70)
print("STRUCTURAL VALIDATION RESULT")
print("=" * 70)

print(
    f"Odd-length histories : {odd_length_count:,}"
)

print(
    f"Status-code mismatches: {mismatches:,}"
)

if odd_length_count != 0:
    raise ValueError(
        "Odd-length payment histories detected."
    )

if mismatches != 0:
    print(
        "WARNING: Some rightmost payment-history codes "
        "do not match current delinquency status."
    )
    print(
        "These mismatches must be investigated before "
        "full behavioural feature construction."
    )
else:
    print(
        "STATUS: PASS — Payment-history structure and "
        "current-month alignment are consistent."
    )

PAYMENT HISTORY LENGTH DISTRIBUTION


,history_length,observations
0,48,9840227


Observations with odd history length: 0

RIGHTMOST PAYMENT-HISTORY CODE


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,payment_history_current_code,current_delinquency_status,observations
0,00,00,11553960
1,01,01,179483
2,02,02,71434
3,03,03,45751
4,04,04,33947
5,05,05,29051
6,06,06,24266
7,07,07,19958
8,08,08,17473
9,09,09,15587


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


PAYMENT HISTORY / CURRENT STATUS AGREEMENT


,observations_checked,exact_matches,mismatches
0,12099114,12099114,0



STRUCTURAL VALIDATION RESULT
Odd-length histories : 0
Status-code mismatches: 0
STATUS: PASS — Payment-history structure and current-month alignment are consistent.


# 9. SQL-Based Payment Behaviour Feature Test

### Objective
This section tests the DuckDB implementation of the payment-history behavioural features before applying it to the complete modelling dataset.

### Why this is necessary
The full modelling dataset contains more than 20 million observations. Applying a row-by-row Python function to the entire dataset would be unnecessarily expensive.

Instead, the behavioural features will be calculated directly in DuckDB.

The current-month payment-history code is removed first. The remaining history therefore represents only information available before month `t`.

The SQL implementation will calculate:

- prior 3-month delinquency indicator;
- prior 6-month delinquency indicator;
- prior 12-month delinquency indicator;
- prior 24-month delinquency indicator;
- maximum prior delinquency over 24 months;
- prior delinquency count over 6 months;
- prior delinquency count over 12 months;
- prior delinquency count over 24 months.

`XX` codes are not counted as delinquency because they do not represent a numeric delinquency status.

This cell only tests the SQL transformation on a small sample.

In [9]:
# ================================================================
# SQL PAYMENT-HISTORY FEATURE TEST
# ================================================================

sql_payment_test_query = f"""
WITH sample AS (

    SELECT
        loan_id,
        reporting_date,
        loan_payment_history,

        /* Remove the current-month two-character code */
        LEFT(
            loan_payment_history,
            LENGTH(loan_payment_history) - 2
        ) AS prior_history

    FROM read_parquet(
        '{BASE_FEATURE_FILE}'
    )

    WHERE loan_payment_history IS NOT NULL
      AND TRIM(loan_payment_history) <> ''

    LIMIT 10
),

windows AS (

    SELECT
        *,

        RIGHT(prior_history, 6)  AS prior_3m,
        RIGHT(prior_history, 12) AS prior_6m,
        RIGHT(prior_history, 24) AS prior_12m,
        RIGHT(prior_history, 48) AS prior_24m

    FROM sample
),

codes AS (

    SELECT
        *,

        regexp_extract_all(
            prior_3m,
            '[0-9]{{2}}'
        ) AS codes_3m,

        regexp_extract_all(
            prior_6m,
            '[0-9]{{2}}'
        ) AS codes_6m,

        regexp_extract_all(
            prior_12m,
            '[0-9]{{2}}'
        ) AS codes_12m,

        regexp_extract_all(
            prior_24m,
            '[0-9]{{2}}'
        ) AS codes_24m

    FROM windows
)

SELECT
    loan_id,
    reporting_date,
    loan_payment_history,

    CASE
        WHEN loan_payment_history IS NULL
            THEN 1
        ELSE 0
    END AS ph_missing,

    CASE
        WHEN list_has_any(
            list_transform(
                codes_3m,
                lambda x : CAST(x AS INTEGER)
            ),
            [1]
        )
        THEN 1
        ELSE
            CASE
                WHEN list_max(
                    list_transform(
                        codes_3m,
                        lambda x : CAST(x AS INTEGER)
                    )
                ) >= 1
                THEN 1
                ELSE 0
            END
    END AS test_prior_3m,

    CASE
        WHEN list_max(
            list_transform(
                codes_6m,
                lambda x : CAST(x AS INTEGER)
            )
        ) >= 1
        THEN 1
        ELSE 0
    END AS test_prior_6m,

    CASE
        WHEN list_max(
            list_transform(
                codes_12m,
                lambda x : CAST(x AS INTEGER)
            )
        ) >= 1
        THEN 1
        ELSE 0
    END AS test_prior_12m,

    CASE
        WHEN list_max(
            list_transform(
                codes_24m,
                lambda x : CAST(x AS INTEGER)
            )
        ) >= 1
        THEN 1
        ELSE 0
    END AS test_prior_24m,

    list_max(
        list_transform(
            codes_24m,
            lambda x : CAST(x AS INTEGER)
        )
    ) AS test_prior_max_24m,

    length(
        list_filter(
            codes_6m,
            lambda x :
                CAST(x AS INTEGER) >= 1
        )
    ) AS test_count_6m,

    length(
        list_filter(
            codes_12m,
            lambda x :
                CAST(x AS INTEGER) >= 1
        )
    ) AS test_count_12m,

    length(
        list_filter(
            codes_24m,
            lambda x :
                CAST(x AS INTEGER) >= 1
        )
    ) AS test_count_24m

FROM codes
"""

sql_payment_test_df = feature_con.execute(
    sql_payment_test_query
).fetchdf()

print("=" * 70)
print("SQL PAYMENT-HISTORY FEATURE TEST")
print("=" * 70)

display(sql_payment_test_df)

SQL PAYMENT-HISTORY FEATURE TEST


,loan_id,reporting_date,loan_payment_history,ph_missing,test_prior_3m,test_prior_6m,test_prior_12m,test_prior_24m,test_prior_max_24m,test_count_6m,test_count_12m,test_count_24m
0,835403805153,2020-06-01,000000000000000000000000000000000000000000000000,0,0,0,0,0,0,0,0,0
1,835410386949,2020-06-01,000000000000000000000000000000000000000000000000,0,0,0,0,0,0,0,0,0
2,835418278429,2020-06-01,000000000000000000000000000000000000000000000000,0,0,0,0,0,0,0,0,0
3,835431738079,2020-06-01,000000010000000000000000000000000000000000000000,0,0,0,0,1,1,0,0,1
4,835434167764,2020-06-01,000000000000000000000000000000000000000000000000,0,0,0,0,0,0,0,0,0
5,835437648684,2020-06-01,000000000000000000000000000000000000000000000000,0,0,0,0,0,0,0,0,0
6,835472578643,2020-06-01,000000000000000000000000000000000000000000000000,0,0,0,0,0,0,0,0,0
7,835475726852,2020-06-01,000000000000000000000000000000000000000000000000,0,0,0,0,0,0,0,0,0
8,835499979283,2020-06-01,000000000000000000000000000000000000000000000000,0,0,0,0,0,0,0,0,0
9,835544366147,2020-06-01,000000000000000000000000000000000000000000000000,0,0,0,0,0,0,0,0,0


## SQL vs Python Payment-History Validation

This cell compares the DuckDB SQL implementation against the independently
tested Python payment-history parser on the same 10 observations.

The SQL and Python results must match exactly for all derived behavioural
features before the full 20M-row feature transformation is executed.

This provides an implementation-level validation that:
- the current-month code is excluded,
- prior 3/6/12/24 month windows are correct,
- delinquency indicators are correct,
- maximum prior delinquency is correct,
- delinquency counts are correct.

In [10]:
# ============================================================
# SQL vs PYTHON PAYMENT-HISTORY FEATURE VALIDATION
# ============================================================

# Re-run the same 10-row SQL sample
sql_validation_df = sql_payment_test_df.copy()

# Apply the independently validated Python parser
python_features = (
    sql_validation_df["loan_payment_history"]
    .apply(parse_payment_history)
    .apply(pd.Series)
)

# Keep only the features that should match SQL
comparison = pd.DataFrame({
    "loan_id": sql_validation_df["loan_id"],
    
    "sql_prior_3m": sql_validation_df["test_prior_3m"],
    "py_prior_3m": python_features["ph_prior_3m_delinquency"],
    
    "sql_prior_6m": sql_validation_df["test_prior_6m"],
    "py_prior_6m": python_features["ph_prior_6m_delinquency"],
    
    "sql_prior_12m": sql_validation_df["test_prior_12m"],
    "py_prior_12m": python_features["ph_prior_12m_delinquency"],
    
    "sql_prior_24m": sql_validation_df["test_prior_24m"],
    "py_prior_24m": python_features["ph_prior_24m_delinquency"],
    
    "sql_max_24m": sql_validation_df["test_prior_max_24m"],
    "py_max_24m": python_features["ph_prior_max_delinquency"],
    
    "sql_count_6m": sql_validation_df["test_count_6m"],
    "py_count_6m": python_features["ph_prior_delinquency_count_6m"],
    
    "sql_count_12m": sql_validation_df["test_count_12m"],
    "py_count_12m": python_features["ph_prior_delinquency_count_12m"],
    
    "sql_count_24m": sql_validation_df["test_count_24m"],
    "py_count_24m": python_features["ph_prior_delinquency_count_24m"],
})

# Compare values
feature_pairs = [
    ("sql_prior_3m", "py_prior_3m"),
    ("sql_prior_6m", "py_prior_6m"),
    ("sql_prior_12m", "py_prior_12m"),
    ("sql_prior_24m", "py_prior_24m"),
    ("sql_max_24m", "py_max_24m"),
    ("sql_count_6m", "py_count_6m"),
    ("sql_count_12m", "py_count_12m"),
    ("sql_count_24m", "py_count_24m"),
]

mismatch_summary = []

for sql_col, py_col in feature_pairs:
    sql_vals = comparison[sql_col]
    py_vals = comparison[py_col]
    
    matches = np.isclose(
        sql_vals.fillna(-999).astype(float),
        py_vals.fillna(-999).astype(float),
        equal_nan=True
    )
    
    mismatch_summary.append({
        "feature": sql_col.replace("sql_", ""),
        "matches": int(matches.sum()),
        "total": len(matches),
        "mismatches": int((~matches).sum())
    })

mismatch_df = pd.DataFrame(mismatch_summary)

print("=" * 70)
print("SQL vs PYTHON PAYMENT-HISTORY VALIDATION")
print("=" * 70)

display(mismatch_df)

total_mismatches = mismatch_df["mismatches"].sum()

print("\nTotal mismatches:", total_mismatches)

if total_mismatches == 0:
    print("PASS: SQL and Python payment-history features match exactly.")
else:
    print("FAIL: SQL and Python results do not match.")
    print("\nRows with mismatches:")
    
    mismatch_mask = np.zeros(len(comparison), dtype=bool)
    
    for sql_col, py_col in feature_pairs:
        mismatch_mask |= ~np.isclose(
            comparison[sql_col].fillna(-999).astype(float),
            comparison[py_col].fillna(-999).astype(float),
            equal_nan=True
        )
    
    display(comparison[mismatch_mask])

SQL vs PYTHON PAYMENT-HISTORY VALIDATION


,feature,matches,total,mismatches
0,prior_3m,10,10,0
1,prior_6m,10,10,0
2,prior_12m,10,10,0
3,prior_24m,10,10,0
4,max_24m,10,10,0
5,count_6m,10,10,0
6,count_12m,10,10,0
7,count_24m,10,10,0



Total mismatches: 0
PASS: SQL and Python payment-history features match exactly.


## Full Payment-History Feature Transformation

This step applies the validated payment-history logic to the complete
20M+ observation modelling dataset.

The current-month payment-history code is deliberately excluded because
the current delinquency status is used only for target eligibility and
must not enter the predictor set.

The following behavioural predictors are created from prior months:

1. ph_missing
2. ph_prior_3m_delinquency
3. ph_prior_6m_delinquency
4. ph_prior_12m_delinquency
5. ph_prior_24m_delinquency
6. ph_prior_max_delinquency
7. ph_prior_delinquency_count_6m
8. ph_prior_delinquency_count_12m
9. ph_prior_delinquency_count_24m

`XX` payment-history codes are treated as unknown and are not counted
as delinquency.

The transformation is executed in DuckDB and written directly to
Parquet to avoid loading the complete dataset into memory.

In [11]:
# ============================================================
# CELL 11 — FULL PAYMENT-HISTORY FEATURE TRANSFORMATION
# ============================================================

from pathlib import Path
import os
import gc

PAYMENT_FEATURE_FILE = (
    FEATURE_OUTPUT_DIR /
    "fannie_2018Q1_payment_features.parquet"
)

print("=" * 70)
print("FULL PAYMENT-HISTORY FEATURE TRANSFORMATION")
print("=" * 70)

# Remove previous output only if it exists
if PAYMENT_FEATURE_FILE.exists():
    PAYMENT_FEATURE_FILE.unlink()

payment_feature_query = f"""
COPY (
    WITH base AS (
        SELECT
            *,
            CASE
                WHEN loan_payment_history IS NULL
                     OR TRIM(loan_payment_history) = ''
                THEN NULL
                ELSE LEFT(
                    loan_payment_history,
                    LENGTH(loan_payment_history) - 2
                )
            END AS prior_history
        FROM read_parquet('{BASE_FEATURE_FILE}')
    ),

    windows AS (
        SELECT
            *,
            RIGHT(prior_history, 6)  AS prior_3m,
            RIGHT(prior_history, 12) AS prior_6m,
            RIGHT(prior_history, 24) AS prior_12m,
            RIGHT(prior_history, 48) AS prior_24m
        FROM base
    ),

    codes AS (
        SELECT
            *,
            regexp_extract_all(
                prior_3m, '[0-9]{{2}}'
            ) AS codes_3m,

            regexp_extract_all(
                prior_6m, '[0-9]{{2}}'
            ) AS codes_6m,

            regexp_extract_all(
                prior_12m, '[0-9]{{2}}'
            ) AS codes_12m,

            regexp_extract_all(
                prior_24m, '[0-9]{{2}}'
            ) AS codes_24m
        FROM windows
    )

    SELECT
        -- Original modelling variables
        loan_id,
        reporting_date,
        target_6m,

        channel,
        seller_name,
        servicer_name,
        original_interest_rate,
        current_interest_rate,
        original_upb,
        current_actual_upb,
        original_loan_term,
        origination_date,
        first_payment_date,
        loan_age,
        remaining_months_legal_maturity,
        remaining_months_maturity,
        maturity_date,
        original_ltv,
        original_cltv,
        number_of_borrowers,
        dti,
        borrower_credit_score,
        co_borrower_credit_score,
        first_time_home_buyer,
        loan_purpose,
        property_type,
        number_of_units,
        occupancy_status,
        property_state,
        msa,
        zip_code_short,
        mortgage_insurance_percentage,
        amortization_type,
        prepayment_penalty_indicator,
        interest_only_indicator,
        modification_flag,

        -- Payment-history missingness
        CASE
            WHEN loan_payment_history IS NULL
                 OR TRIM(loan_payment_history) = ''
            THEN 1
            ELSE 0
        END AS ph_missing,

        -- Prior 3-month delinquency indicator
        CASE
            WHEN COALESCE(
                list_max(
                    list_transform(
                        codes_3m,
                        lambda x : CAST(x AS INTEGER)
                    )
                ),
                0
            ) >= 1
            THEN 1
            ELSE 0
        END AS ph_prior_3m_delinquency,

        -- Prior 6-month delinquency indicator
        CASE
            WHEN COALESCE(
                list_max(
                    list_transform(
                        codes_6m,
                        lambda x : CAST(x AS INTEGER)
                    )
                ),
                0
            ) >= 1
            THEN 1
            ELSE 0
        END AS ph_prior_6m_delinquency,

        -- Prior 12-month delinquency indicator
        CASE
            WHEN COALESCE(
                list_max(
                    list_transform(
                        codes_12m,
                        lambda x : CAST(x AS INTEGER)
                    )
                ),
                0
            ) >= 1
            THEN 1
            ELSE 0
        END AS ph_prior_12m_delinquency,

        -- Prior 24-month delinquency indicator
        CASE
            WHEN COALESCE(
                list_max(
                    list_transform(
                        codes_24m,
                        lambda x : CAST(x AS INTEGER)
                    )
                ),
                0
            ) >= 1
            THEN 1
            ELSE 0
        END AS ph_prior_24m_delinquency,

        -- Maximum delinquency level in prior 24 months
        COALESCE(
            list_max(
                list_transform(
                    codes_24m,
                    lambda x : CAST(x AS INTEGER)
                )
            ),
            0
        ) AS ph_prior_max_delinquency,

        -- Delinquency count in prior 6 months
        length(
            list_filter(
                codes_6m,
                lambda x : CAST(x AS INTEGER) >= 1
            )
        ) AS ph_prior_delinquency_count_6m,

        -- Delinquency count in prior 12 months
        length(
            list_filter(
                codes_12m,
                lambda x : CAST(x AS INTEGER) >= 1
            )
        ) AS ph_prior_delinquency_count_12m,

        -- Delinquency count in prior 24 months
        length(
            list_filter(
                codes_24m,
                lambda x : CAST(x AS INTEGER) >= 1
            )
        ) AS ph_prior_delinquency_count_24m

    FROM codes
)
TO '{PAYMENT_FEATURE_FILE}'
(FORMAT PARQUET, COMPRESSION ZSTD);
"""

print("Source:", BASE_FEATURE_FILE)
print("Output:", PAYMENT_FEATURE_FILE)
print()
print("Starting full transformation...")
print("This may take some time because ~20.15M rows are being processed.")

feature_con.execute(payment_feature_query)

print()
print("Transformation completed.")
print("Output exists:", PAYMENT_FEATURE_FILE.exists())

if PAYMENT_FEATURE_FILE.exists():
    size_mb = PAYMENT_FEATURE_FILE.stat().st_size / (1024 ** 2)
    print(f"Output size: {size_mb:,.2f} MB")

print("=" * 70)

FULL PAYMENT-HISTORY FEATURE TRANSFORMATION
Source: C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning\data\processed\feature_engineering\fannie_2018Q1_base_modelling_dataset.parquet
Output: C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning\data\processed\feature_engineering\fannie_2018Q1_payment_features.parquet

Starting full transformation...
This may take some time because ~20.15M rows are being processed.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Transformation completed.
Output exists: True
Output size: 449.61 MB


## Full Payment-Feature Dataset Validation

This cell validates the completed payment-history transformation before
proceeding to additional feature engineering.

The validation checks:
- total observation count,
- unique loan count,
- target event/non-event counts,
- reporting-date range,
- required columns,
- duplicate loan-month observations,
- payment-history feature ranges,
- consistency of the payment-history missingness indicator,
- absence of NULL values in count-based features.

No modelling is performed at this stage.

In [12]:
# ============================================================
# CELL 12 — FULL PAYMENT-FEATURE DATASET VALIDATION
# ============================================================

print("=" * 70)
print("FULL PAYMENT-FEATURE DATASET VALIDATION")
print("=" * 70)

validation_query = f"""
WITH data AS (
    SELECT *
    FROM read_parquet('{PAYMENT_FEATURE_FILE}')
)

SELECT
    COUNT(*) AS observations,
    COUNT(DISTINCT loan_id) AS unique_loans,
    SUM(CASE WHEN target_6m = 1 THEN 1 ELSE 0 END) AS events,
    SUM(CASE WHEN target_6m = 0 THEN 1 ELSE 0 END) AS non_events,
    COUNT(DISTINCT reporting_date) AS reporting_months,
    MIN(reporting_date) AS min_reporting_date,
    MAX(reporting_date) AS max_reporting_date,

    SUM(CASE WHEN loan_id IS NULL THEN 1 ELSE 0 END) AS null_loan_id,
    SUM(CASE WHEN reporting_date IS NULL THEN 1 ELSE 0 END) AS null_reporting_date,
    SUM(CASE WHEN target_6m IS NULL THEN 1 ELSE 0 END) AS null_target,

    SUM(CASE WHEN ph_missing IS NULL THEN 1 ELSE 0 END) AS null_ph_missing,
    SUM(CASE WHEN ph_prior_3m_delinquency IS NULL THEN 1 ELSE 0 END) AS null_prior_3m,
    SUM(CASE WHEN ph_prior_6m_delinquency IS NULL THEN 1 ELSE 0 END) AS null_prior_6m,
    SUM(CASE WHEN ph_prior_12m_delinquency IS NULL THEN 1 ELSE 0 END) AS null_prior_12m,
    SUM(CASE WHEN ph_prior_24m_delinquency IS NULL THEN 1 ELSE 0 END) AS null_prior_24m,
    SUM(CASE WHEN ph_prior_max_delinquency IS NULL THEN 1 ELSE 0 END) AS null_prior_max,
    SUM(CASE WHEN ph_prior_delinquency_count_6m IS NULL THEN 1 ELSE 0 END) AS null_count_6m,
    SUM(CASE WHEN ph_prior_delinquency_count_12m IS NULL THEN 1 ELSE 0 END) AS null_count_12m,
    SUM(CASE WHEN ph_prior_delinquency_count_24m IS NULL THEN 1 ELSE 0 END) AS null_count_24m

FROM data
"""

validation_df = feature_con.execute(validation_query).fetchdf()

display(validation_df)

# ------------------------------------------------------------
# Duplicate loan-month check
# ------------------------------------------------------------

duplicate_query = f"""
SELECT
    COUNT(*) AS duplicate_loan_month_groups
FROM (
    SELECT
        loan_id,
        reporting_date,
        COUNT(*) AS n
    FROM read_parquet('{PAYMENT_FEATURE_FILE}')
    GROUP BY loan_id, reporting_date
    HAVING COUNT(*) > 1
)
"""

duplicate_df = feature_con.execute(duplicate_query).fetchdf()

duplicate_groups = int(
    duplicate_df.loc[0, "duplicate_loan_month_groups"]
)

print("\nDuplicate loan-month groups:", duplicate_groups)

# ------------------------------------------------------------
# Payment-feature ranges
# ------------------------------------------------------------

range_query = f"""
SELECT
    MIN(ph_missing) AS min_ph_missing,
    MAX(ph_missing) AS max_ph_missing,

    MIN(ph_prior_3m_delinquency) AS min_prior_3m,
    MAX(ph_prior_3m_delinquency) AS max_prior_3m,

    MIN(ph_prior_6m_delinquency) AS min_prior_6m,
    MAX(ph_prior_6m_delinquency) AS max_prior_6m,

    MIN(ph_prior_12m_delinquency) AS min_prior_12m,
    MAX(ph_prior_12m_delinquency) AS max_prior_12m,

    MIN(ph_prior_24m_delinquency) AS min_prior_24m,
    MAX(ph_prior_24m_delinquency) AS max_prior_24m,

    MIN(ph_prior_max_delinquency) AS min_prior_max,
    MAX(ph_prior_max_delinquency) AS max_prior_max,

    MIN(ph_prior_delinquency_count_6m) AS min_count_6m,
    MAX(ph_prior_delinquency_count_6m) AS max_count_6m,

    MIN(ph_prior_delinquency_count_12m) AS min_count_12m,
    MAX(ph_prior_delinquency_count_12m) AS max_count_12m,

    MIN(ph_prior_delinquency_count_24m) AS min_count_24m,
    MAX(ph_prior_delinquency_count_24m) AS max_count_24m
FROM read_parquet('{PAYMENT_FEATURE_FILE}')
"""

range_df = feature_con.execute(range_query).fetchdf()

print("\nPayment-feature ranges:")
display(range_df)

# ------------------------------------------------------------
# Missingness consistency check
# ------------------------------------------------------------

missing_consistency_query = f"""
SELECT
    SUM(
        CASE
            WHEN ph_missing = 1
                 AND loan_payment_history IS NULL
            THEN 0
            WHEN ph_missing = 0
                 AND loan_payment_history IS NOT NULL
            THEN 0
            ELSE 1
        END
    ) AS missingness_mismatches
FROM read_parquet('{PAYMENT_FEATURE_FILE}')
"""

missing_consistency_df = feature_con.execute(
    missing_consistency_query
).fetchdf()

missingness_mismatches = int(
    missing_consistency_df.loc[0, "missingness_mismatches"]
)

print(
    "\nPayment-history missingness mismatches:",
    missingness_mismatches
)

# ------------------------------------------------------------
# Final PASS / FAIL logic
# ------------------------------------------------------------

row_count_ok = int(validation_df.loc[0, "observations"]) == 20_155_269
loan_count_ok = int(validation_df.loc[0, "unique_loans"]) == 450_106
event_count_ok = int(validation_df.loc[0, "events"]) == 646_957
nonevent_count_ok = int(validation_df.loc[0, "non_events"]) == 19_508_312

null_target_ok = int(validation_df.loc[0, "null_target"]) == 0
null_ph_ok = (
    int(validation_df.loc[0, "null_ph_missing"]) == 0
)

duplicate_ok = duplicate_groups == 0
missingness_ok = missingness_mismatches == 0

binary_range_ok = (
    int(range_df.loc[0, "min_prior_3m"]) >= 0 and
    int(range_df.loc[0, "max_prior_3m"]) <= 1 and
    int(range_df.loc[0, "min_prior_6m"]) >= 0 and
    int(range_df.loc[0, "max_prior_6m"]) <= 1 and
    int(range_df.loc[0, "min_prior_12m"]) >= 0 and
    int(range_df.loc[0, "max_prior_12m"]) <= 1 and
    int(range_df.loc[0, "min_prior_24m"]) >= 0 and
    int(range_df.loc[0, "max_prior_24m"]) <= 1
)

count_range_ok = (
    int(range_df.loc[0, "min_count_6m"]) >= 0 and
    int(range_df.loc[0, "max_count_6m"]) <= 3 and
    int(range_df.loc[0, "min_count_12m"]) >= 0 and
    int(range_df.loc[0, "max_count_12m"]) <= 6 and
    int(range_df.loc[0, "min_count_24m"]) >= 0 and
    int(range_df.loc[0, "max_count_24m"]) <= 12
)

print("\n" + "=" * 70)
print("VALIDATION RESULT")
print("=" * 70)

checks = {
    "Row count": row_count_ok,
    "Unique loans": loan_count_ok,
    "Event count": event_count_ok,
    "Non-event count": nonevent_count_ok,
    "No NULL targets": null_target_ok,
    "No NULL ph_missing": null_ph_ok,
    "No duplicate loan-months": duplicate_ok,
    "Missingness consistency": missingness_ok,
    "Binary feature ranges": binary_range_ok,
    "Count feature ranges": count_range_ok,
}

for name, passed in checks.items():
    print(f"{name:<30}: {'PASS' if passed else 'FAIL'}")

overall_pass = all(checks.values())

print("\nOVERALL:", "PASS" if overall_pass else "FAIL")
print("=" * 70)

FULL PAYMENT-FEATURE DATASET VALIDATION


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,observations,unique_loans,events,non_events,reporting_months,min_reporting_date,max_reporting_date,null_loan_id,null_reporting_date,null_target,null_ph_missing,null_prior_3m,null_prior_6m,null_prior_12m,null_prior_24m,null_prior_max,null_count_6m,null_count_12m,null_count_24m
0,20155269,450106,646957.0,19508312.0,93,2018-01-01,2025-09-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10315042.0,10315042.0,10315042.0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Duplicate loan-month groups: 0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Payment-feature ranges:


,min_ph_missing,max_ph_missing,min_prior_3m,max_prior_3m,min_prior_6m,max_prior_6m,min_prior_12m,max_prior_12m,min_prior_24m,max_prior_24m,min_prior_max,max_prior_max,min_count_6m,max_count_6m,min_count_12m,max_count_12m,min_count_24m,max_count_24m
0,0,1,0,1,0,1,0,1,0,1,0,79,0,6,0,12,0,23


BinderException: Binder Error: Referenced column "loan_payment_history" not found in FROM clause!
Candidate bindings: "loan_age", "loan_id", "loan_purpose", "remaining_months_maturity", "original_loan_term"

LINE 6:                  AND loan_payment_history IS NULL
                             ^

In [13]:
# ============================================================
# CELL 12B — CORRECTED PAYMENT-FEATURE VALIDATION
# ============================================================

print("=" * 70)
print("CORRECTED PAYMENT-FEATURE VALIDATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. Validate output-level structure
# ------------------------------------------------------------

structure_query = f"""
SELECT
    COUNT(*) AS observations,
    COUNT(DISTINCT loan_id) AS unique_loans,
    SUM(CASE WHEN target_6m = 1 THEN 1 ELSE 0 END) AS events,
    SUM(CASE WHEN target_6m = 0 THEN 1 ELSE 0 END) AS non_events,
    COUNT(DISTINCT reporting_date) AS reporting_months,
    MIN(reporting_date) AS min_reporting_date,
    MAX(reporting_date) AS max_reporting_date
FROM read_parquet('{PAYMENT_FEATURE_FILE}')
"""

structure_df = feature_con.execute(structure_query).fetchdf()

display(structure_df)

# ------------------------------------------------------------
# 2. Validate payment-history missingness against SOURCE
# ------------------------------------------------------------

source_missing_query = f"""
SELECT
    COUNT(*) AS source_observations,
    SUM(
        CASE
            WHEN loan_payment_history IS NULL
                 OR TRIM(loan_payment_history) = ''
            THEN 1
            ELSE 0
        END
    ) AS source_missing_history
FROM read_parquet('{BASE_FEATURE_FILE}')
"""

source_missing_df = feature_con.execute(
    source_missing_query
).fetchdf()

# ------------------------------------------------------------
# 3. Validate generated ph_missing
# ------------------------------------------------------------

output_missing_query = f"""
SELECT
    COUNT(*) AS output_observations,
    SUM(CASE WHEN ph_missing = 1 THEN 1 ELSE 0 END)
        AS output_missing_history
FROM read_parquet('{PAYMENT_FEATURE_FILE}')
"""

output_missing_df = feature_con.execute(
    output_missing_query
).fetchdf()

source_obs = int(source_missing_df.loc[0, "source_observations"])
source_missing = int(
    source_missing_df.loc[0, "source_missing_history"]
)

output_obs = int(output_missing_df.loc[0, "output_observations"])
output_missing = int(
    output_missing_df.loc[0, "output_missing_history"]
)

print("\nPayment-history missingness comparison:")
print(f"Source observations : {source_obs:,}")
print(f"Output observations : {output_obs:,}")
print(f"Source missing      : {source_missing:,}")
print(f"Output ph_missing=1 : {output_missing:,}")

# ------------------------------------------------------------
# 4. Validate generated feature ranges
# ------------------------------------------------------------

range_query = f"""
SELECT
    MIN(ph_missing) AS min_ph_missing,
    MAX(ph_missing) AS max_ph_missing,

    MIN(ph_prior_3m_delinquency) AS min_prior_3m,
    MAX(ph_prior_3m_delinquency) AS max_prior_3m,

    MIN(ph_prior_6m_delinquency) AS min_prior_6m,
    MAX(ph_prior_6m_delinquency) AS max_prior_6m,

    MIN(ph_prior_12m_delinquency) AS min_prior_12m,
    MAX(ph_prior_12m_delinquency) AS max_prior_12m,

    MIN(ph_prior_24m_delinquency) AS min_prior_24m,
    MAX(ph_prior_24m_delinquency) AS max_prior_24m,

    MIN(ph_prior_max_delinquency) AS min_prior_max,
    MAX(ph_prior_max_delinquency) AS max_prior_max,

    MIN(ph_prior_delinquency_count_6m) AS min_count_6m,
    MAX(ph_prior_delinquency_count_6m) AS max_count_6m,

    MIN(ph_prior_delinquency_count_12m) AS min_count_12m,
    MAX(ph_prior_delinquency_count_12m) AS max_count_12m,

    MIN(ph_prior_delinquency_count_24m) AS min_count_24m,
    MAX(ph_prior_delinquency_count_24m) AS max_count_24m,

    SUM(CASE WHEN ph_missing = 1 THEN 1 ELSE 0 END)
        AS missing_history_rows,

    SUM(
        CASE
            WHEN ph_missing = 1
                 AND (
                     ph_prior_3m_delinquency <> 0
                     OR ph_prior_6m_delinquency <> 0
                     OR ph_prior_12m_delinquency <> 0
                     OR ph_prior_24m_delinquency <> 0
                     OR ph_prior_max_delinquency <> 0
                     OR ph_prior_delinquency_count_6m <> 0
                     OR ph_prior_delinquency_count_12m <> 0
                     OR ph_prior_delinquency_count_24m <> 0
                 )
            THEN 1
            ELSE 0
        END
    ) AS missing_history_feature_mismatches

FROM read_parquet('{PAYMENT_FEATURE_FILE}')
"""

range_df = feature_con.execute(range_query).fetchdf()

print("\nPayment-feature validation:")
display(range_df)

# ------------------------------------------------------------
# 5. Duplicate loan-month validation
# ------------------------------------------------------------

duplicate_query = f"""
SELECT COUNT(*) AS duplicate_loan_month_groups
FROM (
    SELECT
        loan_id,
        reporting_date,
        COUNT(*) AS n
    FROM read_parquet('{PAYMENT_FEATURE_FILE}')
    GROUP BY loan_id, reporting_date
    HAVING COUNT(*) > 1
)
"""

duplicate_df = feature_con.execute(duplicate_query).fetchdf()

duplicate_groups = int(
    duplicate_df.loc[0, "duplicate_loan_month_groups"]
)

print("\nDuplicate loan-month groups:", duplicate_groups)

# ------------------------------------------------------------
# 6. PASS / FAIL
# ------------------------------------------------------------

row_count_ok = output_obs == 20_155_269
loan_count_ok = int(structure_df.loc[0, "unique_loans"]) == 450_106
event_count_ok = int(structure_df.loc[0, "events"]) == 646_957
nonevent_count_ok = int(structure_df.loc[0, "non_events"]) == 19_508_312

missing_count_ok = (
    source_missing == output_missing
    and source_obs == output_obs
)

missing_feature_ok = (
    int(range_df.loc[0, "missing_history_feature_mismatches"]) == 0
)

duplicate_ok = duplicate_groups == 0

binary_range_ok = (
    int(range_df.loc[0, "min_prior_3m"]) >= 0
    and int(range_df.loc[0, "max_prior_3m"]) <= 1
    and int(range_df.loc[0, "min_prior_6m"]) >= 0
    and int(range_df.loc[0, "max_prior_6m"]) <= 1
    and int(range_df.loc[0, "min_prior_12m"]) >= 0
    and int(range_df.loc[0, "max_prior_12m"]) <= 1
    and int(range_df.loc[0, "min_prior_24m"]) >= 0
    and int(range_df.loc[0, "max_prior_24m"]) <= 1
)

count_range_ok = (
    int(range_df.loc[0, "min_count_6m"]) >= 0
    and int(range_df.loc[0, "max_count_6m"]) <= 3
    and int(range_df.loc[0, "min_count_12m"]) >= 0
    and int(range_df.loc[0, "max_count_12m"]) <= 6
    and int(range_df.loc[0, "min_count_24m"]) >= 0
    and int(range_df.loc[0, "max_count_24m"]) <= 12
)

print("\n" + "=" * 70)
print("VALIDATION RESULT")
print("=" * 70)

checks = {
    "Row count": row_count_ok,
    "Unique loans": loan_count_ok,
    "Event count": event_count_ok,
    "Non-event count": nonevent_count_ok,
    "Missing-history count": missing_count_ok,
    "Missing-history features": missing_feature_ok,
    "No duplicate loan-months": duplicate_ok,
    "Binary feature ranges": binary_range_ok,
    "Count feature ranges": count_range_ok,
}

for name, passed in checks.items():
    print(f"{name:<30}: {'PASS' if passed else 'FAIL'}")

overall_pass = all(checks.values())

print("\nOVERALL:", "PASS" if overall_pass else "FAIL")
print("=" * 70)

CORRECTED PAYMENT-FEATURE VALIDATION


,observations,unique_loans,events,non_events,reporting_months,min_reporting_date,max_reporting_date
0,20155269,450106,646957.0,19508312.0,93,2018-01-01,2025-09-01



Payment-history missingness comparison:
Source observations : 20,155,269
Output observations : 20,155,269
Source missing      : 10,315,042
Output ph_missing=1 : 10,315,042

Payment-feature validation:


,min_ph_missing,max_ph_missing,min_prior_3m,max_prior_3m,min_prior_6m,max_prior_6m,min_prior_12m,max_prior_12m,min_prior_24m,max_prior_24m,min_prior_max,max_prior_max,min_count_6m,max_count_6m,min_count_12m,max_count_12m,min_count_24m,max_count_24m,missing_history_rows,missing_history_feature_mismatches
0,0,1,0,1,0,1,0,1,0,1,0,79,0,6,0,12,0,23,10315042.0,0.0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Duplicate loan-month groups: 0

VALIDATION RESULT
Row count                     : PASS
Unique loans                  : PASS
Event count                   : PASS
Non-event count               : PASS
Missing-history count         : PASS
Missing-history features      : PASS
No duplicate loan-months      : PASS
Binary feature ranges         : PASS
Count feature ranges          : FAIL

OVERALL: FAIL


In [14]:
# ============================================================
# CELL 12C — CORRECTED PAYMENT-COUNT RANGE VALIDATION
# ============================================================

print("=" * 70)
print("CORRECTED PAYMENT-COUNT RANGE VALIDATION")
print("=" * 70)

count_range_query = f"""
SELECT
    MIN(ph_prior_delinquency_count_6m) AS min_count_6m,
    MAX(ph_prior_delinquency_count_6m) AS max_count_6m,

    MIN(ph_prior_delinquency_count_12m) AS min_count_12m,
    MAX(ph_prior_delinquency_count_12m) AS max_count_12m,

    MIN(ph_prior_delinquency_count_24m) AS min_count_24m,
    MAX(ph_prior_delinquency_count_24m) AS max_count_24m
FROM read_parquet('{PAYMENT_FEATURE_FILE}')
"""

count_range_df = feature_con.execute(count_range_query).fetchdf()

display(count_range_df)

count_range_ok = (
    int(count_range_df.loc[0, "min_count_6m"]) >= 0
    and int(count_range_df.loc[0, "max_count_6m"]) <= 6

    and int(count_range_df.loc[0, "min_count_12m"]) >= 0
    and int(count_range_df.loc[0, "max_count_12m"]) <= 12

    and int(count_range_df.loc[0, "min_count_24m"]) >= 0
    and int(count_range_df.loc[0, "max_count_24m"]) <= 24
)

print("\nCorrect count limits:")
print("6-month  : 0–6")
print("12-month : 0–12")
print("24-month : 0–24")

print(
    "\nCount feature ranges:",
    "PASS" if count_range_ok else "FAIL"
)

print("=" * 70)

CORRECTED PAYMENT-COUNT RANGE VALIDATION


,min_count_6m,max_count_6m,min_count_12m,max_count_12m,min_count_24m,max_count_24m
0,0,6,0,12,0,23



Correct count limits:
6-month  : 0–6
12-month : 0–12
24-month : 0–24

Count feature ranges: PASS


## Financial & Loan-Derived Feature Engineering — Formula Test

This step creates derived borrower, balance, interest-rate, LTV/CLTV,
credit-score, and loan-maturity features using information available at
the observation month.

The formulas are first tested on a small sample before being applied to
the complete modelling dataset.

Candidate derived features:

1. balance_utilization
   = current_actual_upb / original_upb

2. interest_rate_change
   = current_interest_rate - original_interest_rate

3. ltv_cltv_gap
   = original_cltv - original_ltv

4. loan_age_ratio
   = loan_age / original_loan_term

5. remaining_term_ratio
   = remaining_months_maturity / original_loan_term

6. borrower_credit_score_missing

7. co_borrower_credit_score_missing

8. mortgage_insurance_missing

9. co_borrower_score_gap
   = co_borrower_credit_score - borrower_credit_score

10. reporting_year

11. reporting_month

Division-by-zero cases are protected using NULLIF-equivalent logic.

Maturity_date is not used for derived features because the raw-data
validation identified a small number of anomalous maturity dates.

All derived features use information available at time t only.
No future information is used.

In [3]:
# ============================================================
# CELL 13 — FINANCIAL & LOAN-DERIVED FEATURE TEST
# ============================================================

print("=" * 70)
print("FINANCIAL & LOAN-DERIVED FEATURE TEST")
print("=" * 70)

financial_test_query = f"""
WITH sample AS (
    SELECT *
    FROM read_parquet('{PAYMENT_FEATURE_FILE}')
    LIMIT 1000
)

SELECT
    loan_id,
    reporting_date,
    target_6m,

    original_upb,
    current_actual_upb,
    original_interest_rate,
    current_interest_rate,
    original_ltv,
    original_cltv,
    loan_age,
    original_loan_term,
    remaining_months_maturity,
    borrower_credit_score,
    co_borrower_credit_score,
    mortgage_insurance_percentage,

    -- -------------------------------------------------------
    -- 1. Balance utilization
    -- -------------------------------------------------------
    CASE
        WHEN original_upb > 0
        THEN current_actual_upb / original_upb
        ELSE NULL
    END AS balance_utilization,

    -- -------------------------------------------------------
    -- 2. Interest-rate change
    -- -------------------------------------------------------
    CASE
        WHEN current_interest_rate IS NOT NULL
             AND original_interest_rate IS NOT NULL
        THEN current_interest_rate - original_interest_rate
        ELSE NULL
    END AS interest_rate_change,

    -- -------------------------------------------------------
    -- 3. LTV / CLTV gap
    -- -------------------------------------------------------
    CASE
        WHEN original_ltv IS NOT NULL
             AND original_cltv IS NOT NULL
        THEN original_cltv - original_ltv
        ELSE NULL
    END AS ltv_cltv_gap,

    -- -------------------------------------------------------
    -- 4. Loan age relative to original term
    -- -------------------------------------------------------
    CASE
        WHEN original_loan_term > 0
             AND loan_age IS NOT NULL
        THEN loan_age / original_loan_term
        ELSE NULL
    END AS loan_age_ratio,

    -- -------------------------------------------------------
    -- 5. Remaining maturity relative to original term
    -- -------------------------------------------------------
    CASE
        WHEN original_loan_term > 0
             AND remaining_months_maturity IS NOT NULL
        THEN remaining_months_maturity / original_loan_term
        ELSE NULL
    END AS remaining_term_ratio,

    -- -------------------------------------------------------
    -- 6–8. Missingness indicators
    -- -------------------------------------------------------
    CASE
        WHEN borrower_credit_score IS NULL THEN 1
        ELSE 0
    END AS borrower_credit_score_missing,

    CASE
        WHEN co_borrower_credit_score IS NULL THEN 1
        ELSE 0
    END AS co_borrower_credit_score_missing,

    CASE
        WHEN mortgage_insurance_percentage IS NULL THEN 1
        ELSE 0
    END AS mortgage_insurance_missing,

    -- -------------------------------------------------------
    -- 9. Co-borrower score gap
    -- -------------------------------------------------------
    CASE
        WHEN borrower_credit_score IS NOT NULL
             AND co_borrower_credit_score IS NOT NULL
        THEN co_borrower_credit_score - borrower_credit_score
        ELSE NULL
    END AS co_borrower_score_gap,

    -- -------------------------------------------------------
    -- 10–11. Calendar features
    -- -------------------------------------------------------
    EXTRACT(YEAR FROM reporting_date) AS reporting_year,

    EXTRACT(MONTH FROM reporting_date) AS reporting_month

FROM sample
"""

financial_test_df = feature_con.execute(
    financial_test_query
).fetchdf()

print(f"Test observations: {len(financial_test_df):,}")
print()

display(financial_test_df.head(10))

print("\nDerived-feature summary:")

display(
    financial_test_df[
        [
            "balance_utilization",
            "interest_rate_change",
            "ltv_cltv_gap",
            "loan_age_ratio",
            "remaining_term_ratio",
            "co_borrower_score_gap",
            "reporting_year",
            "reporting_month"
        ]
    ].describe()
)

print("\nMissingness indicators:")

display(
    financial_test_df[
        [
            "borrower_credit_score_missing",
            "co_borrower_credit_score_missing",
            "mortgage_insurance_missing"
        ]
    ].sum().to_frame("count")
)

print("\nFormula/range checks:")

checks = {
    "balance_utilization >= 0":
        financial_test_df["balance_utilization"]
        .dropna()
        .ge(0)
        .all(),

    "balance_utilization finite":
        np.isfinite(
            financial_test_df["balance_utilization"].dropna()
        ).all(),

    "interest_rate_change finite":
        np.isfinite(
            financial_test_df["interest_rate_change"].dropna()
        ).all(),

    "ltv_cltv_gap finite":
        np.isfinite(
            financial_test_df["ltv_cltv_gap"].dropna()
        ).all(),

    "loan_age_ratio finite":
        np.isfinite(
            financial_test_df["loan_age_ratio"].dropna()
        ).all(),

    "remaining_term_ratio finite":
        np.isfinite(
            financial_test_df["remaining_term_ratio"].dropna()
        ).all(),

    "reporting_month 1-12":
        financial_test_df["reporting_month"]
        .between(1, 12)
        .all(),

    "missing indicators binary":
        all(
            set(
                financial_test_df[col]
                .dropna()
                .unique()
            ).issubset({0, 1})
            for col in [
                "borrower_credit_score_missing",
                "co_borrower_credit_score_missing",
                "mortgage_insurance_missing"
            ]
        )
}

for name, passed in checks.items():
    print(
        f"{name:<35}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

overall_pass = all(checks.values())

print("\n" + "=" * 70)
print(
    "OVERALL:",
    "PASS" if overall_pass else "FAIL"
)
print("=" * 70)

FINANCIAL & LOAN-DERIVED FEATURE TEST
Test observations: 1,000



,loan_id,reporting_date,target_6m,original_upb,current_actual_upb,original_interest_rate,current_interest_rate,original_ltv,original_cltv,loan_age,...,interest_rate_change,ltv_cltv_gap,loan_age_ratio,remaining_term_ratio,borrower_credit_score_missing,co_borrower_credit_score_missing,mortgage_insurance_missing,co_borrower_score_gap,reporting_year,reporting_month
0,100022479643,2018-01-01,0,200000.0,0.0,4.000,4.000,80,80.0,1.0,...,0.0,0.0,0.002778,0.994444,0,1,1,NaN,2018,1
1,100042858582,2018-01-01,0,277000.0,0.0,3.875,3.875,86,92.0,0.0,...,0.0,6.0,0.000000,0.995833,0,0,0,8.0,2018,1
2,100187480788,2018-01-01,0,261000.0,0.0,4.000,4.000,80,80.0,1.0,...,0.0,0.0,0.002778,0.997222,0,0,1,17.0,2018,1
3,100204823543,2018-01-01,0,441000.0,0.0,4.375,4.375,95,95.0,1.0,...,0.0,0.0,0.002778,0.997222,0,0,0,0.0,2018,1
4,100283566127,2018-01-01,0,349000.0,0.0,4.500,4.500,97,97.0,0.0,...,0.0,0.0,0.000000,1.000000,0,1,0,NaN,2018,1
5,100285479030,2018-01-01,0,124000.0,0.0,4.250,4.250,95,95.0,0.0,...,0.0,0.0,0.000000,1.000000,0,0,0,-2.0,2018,1
6,100300458167,2018-01-01,0,200000.0,0.0,4.000,4.000,95,95.0,0.0,...,0.0,0.0,0.000000,1.000000,0,0,0,-9.0,2018,1
7,100300675626,2018-01-01,0,311000.0,0.0,4.625,4.625,75,75.0,0.0,...,0.0,0.0,0.000000,1.000000,0,0,1,-130.0,2018,1
8,100360680170,2018-01-01,0,213000.0,0.0,3.250,3.250,55,55.0,0.0,...,0.0,0.0,0.000000,1.000000,0,0,1,-2.0,2018,1
9,100438351680,2018-01-01,0,300000.0,0.0,4.000,4.000,69,69.0,0.0,...,0.0,0.0,0.000000,1.000000,0,0,1,-37.0,2018,1



Derived-feature summary:


,balance_utilization,interest_rate_change,ltv_cltv_gap,loan_age_ratio,remaining_term_ratio,co_borrower_score_gap,reporting_year,reporting_month
count,1000.0,1000.0,1000.000000,1000.000000,1000.000000,427.000000,1000.0,1000.0
mean,0.0,0.0,0.525000,0.000610,0.997454,2.477752,2018.0,1.0
std,0.0,0.0,2.951624,0.002681,0.004775,38.108338,0.0,0.0
min,0.0,0.0,0.000000,-0.008333,0.933333,-130.000000,2018.0,1.0
25%,0.0,0.0,0.000000,0.000000,0.997222,-17.500000,2018.0,1.0
50%,0.0,0.0,0.000000,0.000000,0.997222,1.000000,2018.0,1.0
75%,0.0,0.0,0.000000,0.002778,1.000000,20.000000,2018.0,1.0
max,0.0,0.0,47.000000,0.016667,1.000000,180.000000,2018.0,1.0



Missingness indicators:


,count
borrower_credit_score_missing,0
co_borrower_credit_score_missing,573
mortgage_insurance_missing,694



Formula/range checks:
balance_utilization >= 0           : PASS
balance_utilization finite         : PASS
interest_rate_change finite        : PASS
ltv_cltv_gap finite                : PASS
loan_age_ratio finite              : PASS
remaining_term_ratio finite        : PASS
reporting_month 1-12               : PASS
missing indicators binary          : PASS

OVERALL: PASS


In [2]:
# ============================================================
# CELL 13A — RESTORE FEATURE-ENGINEERING SESSION
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import duckdb
import os

# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------

PROJECT_DIR = Path(
    r"C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning"
)

DATA_DIR = PROJECT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
FEATURE_OUTPUT_DIR = PROCESSED_DIR / "feature_engineering"

PAYMENT_FEATURE_FILE = (
    FEATURE_OUTPUT_DIR /
    "fannie_2018Q1_payment_features.parquet"
)

# ------------------------------------------------------------
# Check existing payment-feature file
# ------------------------------------------------------------

print("=" * 70)
print("RESTORING FEATURE-ENGINEERING SESSION")
print("=" * 70)

print("Payment-feature file:")
print(PAYMENT_FEATURE_FILE)

print("\nFile exists:", PAYMENT_FEATURE_FILE.exists())

if PAYMENT_FEATURE_FILE.exists():
    size_mb = PAYMENT_FEATURE_FILE.stat().st_size / (1024 ** 2)
    print(f"File size: {size_mb:,.2f} MB")
else:
    raise FileNotFoundError(
        "The payment-feature Parquet file was not found. "
        "Do NOT rerun anything yet."
    )

# ------------------------------------------------------------
# Reconnect DuckDB
# ------------------------------------------------------------

FEATURE_TEMP_DIR = FEATURE_OUTPUT_DIR / "duckdb_temp"
FEATURE_TEMP_DIR.mkdir(parents=True, exist_ok=True)

feature_con = duckdb.connect()

feature_con.execute("SET memory_limit = '4GB'")
feature_con.execute("SET threads = 2")
feature_con.execute("SET preserve_insertion_order = false")
feature_con.execute(
    f"SET temp_directory = '{FEATURE_TEMP_DIR}'"
)

print("\nDuckDB version:", duckdb.__version__)
print("Memory limit: 4GB")
print("Threads: 2")
print("Session restored successfully.")

print("=" * 70)

RESTORING FEATURE-ENGINEERING SESSION
Payment-feature file:
C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning\data\processed\feature_engineering\fannie_2018Q1_payment_features.parquet

File exists: True
File size: 449.61 MB

DuckDB version: 1.5.5
Memory limit: 4GB
Threads: 2
Session restored successfully.


## Financial & Loan-Derived Feature — Diverse Sample Validation

The previous formula test passed, but the sample was concentrated in the
earliest reporting month and therefore was not representative of the full
dataset.

This cell selects observations from across the modelling dataset and checks
the derived financial features against their underlying source variables.

The purpose is to confirm that:

- current balances are not systematically represented as zero,
- balance utilization behaves as expected,
- interest-rate changes are correctly calculated,
- LTV/CLTV relationships are sensible,
- loan-age and remaining-term ratios are correctly derived,
- missingness indicators correspond to the underlying variables.

No full-dataset transformation is performed.

In [5]:
# ============================================================
# CELL 14 — DIVERSE FINANCIAL FEATURE SANITY CHECK
# ============================================================

print("=" * 70)
print("DIVERSE FINANCIAL FEATURE SANITY CHECK")
print("=" * 70)

diverse_query = f"""
WITH sampled AS (
    SELECT *
    FROM read_parquet('{PAYMENT_FEATURE_FILE}')
    USING SAMPLE reservoir(2000 ROWS)
)

SELECT
    loan_id,
    reporting_date,

    original_upb,
    current_actual_upb,

    original_interest_rate,
    current_interest_rate,

    original_ltv,
    original_cltv,

    loan_age,
    original_loan_term,
    remaining_months_maturity,

    borrower_credit_score,
    co_borrower_credit_score,
    mortgage_insurance_percentage,

    CASE
        WHEN original_upb > 0
        THEN current_actual_upb / original_upb
        ELSE NULL
    END AS balance_utilization,

    current_interest_rate - original_interest_rate
        AS interest_rate_change,

    original_cltv - original_ltv
        AS ltv_cltv_gap,

    CASE
        WHEN original_loan_term > 0
        THEN loan_age / original_loan_term
        ELSE NULL
    END AS loan_age_ratio,

    CASE
        WHEN original_loan_term > 0
        THEN remaining_months_maturity / original_loan_term
        ELSE NULL
    END AS remaining_term_ratio,

    CASE
        WHEN borrower_credit_score IS NULL THEN 1
        ELSE 0
    END AS borrower_credit_score_missing,

    CASE
        WHEN co_borrower_credit_score IS NULL THEN 1
        ELSE 0
    END AS co_borrower_credit_score_missing,

    CASE
        WHEN mortgage_insurance_percentage IS NULL THEN 1
        ELSE 0
    END AS mortgage_insurance_missing,

    CASE
        WHEN borrower_credit_score IS NOT NULL
             AND co_borrower_credit_score IS NOT NULL
        THEN co_borrower_credit_score - borrower_credit_score
        ELSE NULL
    END AS co_borrower_score_gap

FROM sampled
ORDER BY reporting_date, loan_id
"""

diverse_df = feature_con.execute(diverse_query).fetchdf()

print(f"Sample observations: {len(diverse_df):,}")

print(
    "Reporting dates:",
    diverse_df["reporting_date"].min(),
    "to",
    diverse_df["reporting_date"].max()
)

print("\nNumber of distinct reporting months:")
print(diverse_df["reporting_date"].nunique())

print("\nUnderlying financial variables:")
display(
    diverse_df[
        [
            "reporting_date",
            "original_upb",
            "current_actual_upb",
            "original_interest_rate",
            "current_interest_rate",
            "original_ltv",
            "original_cltv",
            "loan_age",
            "remaining_months_maturity"
        ]
    ].head(15)
)

print("\nDerived financial features:")
display(
    diverse_df[
        [
            "balance_utilization",
            "interest_rate_change",
            "ltv_cltv_gap",
            "loan_age_ratio",
            "remaining_term_ratio",
            "co_borrower_score_gap"
        ]
    ].describe()
)

print("\nBalance-utilization diagnostics:")

balance_diag = pd.DataFrame({
    "metric": [
        "rows",
        "zero_balance_rows",
        "positive_balance_rows",
        "missing_balance_utilization",
        "maximum_balance_utilization"
    ],
    "value": [
        len(diverse_df),
        int((diverse_df["current_actual_upb"] == 0).sum()),
        int((diverse_df["current_actual_upb"] > 0).sum()),
        int(diverse_df["balance_utilization"].isna().sum()),
        diverse_df["balance_utilization"].max()
    ]
})

display(balance_diag)

print("\nFormula checks:")

checks = {
    "Sample is non-empty":
        len(diverse_df) > 0,

    "Multiple reporting months":
        diverse_df["reporting_date"].nunique() > 1,

    "Balance utilization finite":
        np.isfinite(
            diverse_df["balance_utilization"].dropna()
        ).all(),

    "Interest-rate change finite":
        np.isfinite(
            diverse_df["interest_rate_change"].dropna()
        ).all(),

    "LTV/CLTV gap finite":
        np.isfinite(
            diverse_df["ltv_cltv_gap"].dropna()
        ).all(),

    "Loan-age ratio finite":
        np.isfinite(
            diverse_df["loan_age_ratio"].dropna()
        ).all(),

    "Remaining-term ratio finite":
        np.isfinite(
            diverse_df["remaining_term_ratio"].dropna()
        ).all(),

    "Missingness indicators binary":
        all(
            set(diverse_df[col].dropna().unique()).issubset({0, 1})
            for col in [
                "borrower_credit_score_missing",
                "co_borrower_credit_score_missing",
                "mortgage_insurance_missing"
            ]
        )
    }

for name, passed in checks.items():
    print(
        f"{name:<35}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

print("=" * 70)

DIVERSE FINANCIAL FEATURE SANITY CHECK
Sample observations: 2,000
Reporting dates: 2018-01-01 00:00:00 to 2025-09-01 00:00:00

Number of distinct reporting months:
93

Underlying financial variables:


,reporting_date,original_upb,current_actual_upb,original_interest_rate,current_interest_rate,original_ltv,original_cltv,loan_age,remaining_months_maturity
0,2018-01-01,225000.0,0.0,3.990,3.990,55,55.0,0.0,360.0
1,2018-01-01,40000.0,0.0,4.250,4.250,80,80.0,-1.0,360.0
2,2018-01-01,50000.0,0.0,4.375,4.375,74,74.0,0.0,360.0
3,2018-01-01,426000.0,0.0,4.250,4.250,95,95.0,0.0,360.0
4,2018-01-01,110000.0,0.0,4.875,4.875,75,75.0,3.0,356.0
5,2018-01-01,97000.0,0.0,4.375,4.375,46,46.0,1.0,358.0
6,2018-01-01,174000.0,0.0,4.000,4.000,67,67.0,1.0,239.0
7,2018-01-01,424000.0,0.0,3.875,3.875,59,59.0,0.0,359.0
8,2018-01-01,440000.0,0.0,4.375,4.375,80,80.0,-1.0,360.0
9,2018-01-01,260000.0,0.0,3.500,3.500,80,80.0,-1.0,180.0



Derived financial features:


,balance_utilization,interest_rate_change,ltv_cltv_gap,loan_age_ratio,remaining_term_ratio,co_borrower_score_gap
count,2000.000000,2000.000000,2000.00000,2000.000000,1975.000000,853.000000
mean,0.779640,-0.007437,0.61550,0.109338,0.866526,-0.807737
std,0.326028,0.096035,3.16428,0.101275,0.141406,39.064688
min,0.000000,-1.875000,0.00000,-0.008333,0.047222,-156.000000
25%,0.813428,0.000000,0.00000,0.033333,0.811806,-17.000000
50%,0.926664,0.000000,0.00000,0.077778,0.913889,0.000000
75%,0.968387,0.000000,0.00000,0.161111,0.962500,17.000000
max,1.155263,0.000000,54.00000,0.766667,1.000000,169.000000



Balance-utilization diagnostics:


,metric,value
0,rows,2000.000000
1,zero_balance_rows,262.000000
2,positive_balance_rows,1738.000000
3,missing_balance_utilization,0.000000
4,maximum_balance_utilization,1.155263



Formula checks:
Sample is non-empty                : PASS
Multiple reporting months          : PASS
Balance utilization finite         : PASS
Interest-rate change finite        : PASS
LTV/CLTV gap finite                : PASS
Loan-age ratio finite              : PASS
Remaining-term ratio finite        : PASS
Missingness indicators binary      : PASS


## Full Financial & Loan-Derived Feature Transformation

The financial and loan-derived features validated in the previous cells
are now applied to the complete modelling dataset.

All derived variables use information available at the observation month
and therefore do not use future target information.

The following features are added:

1. balance_utilization
2. interest_rate_change
3. ltv_cltv_gap
4. loan_age_ratio
5. remaining_term_ratio
6. borrower_credit_score_missing
7. co_borrower_credit_score_missing
8. mortgage_insurance_missing
9. co_borrower_score_gap
10. reporting_year
11. reporting_month

Maturity_date is retained as a source variable but is not used to create
derived features because the raw-data validation identified a small
number of anomalous maturity-date observations.

No arbitrary clipping or winsorization is applied at this stage.
Observed source-derived values are retained for subsequent modelling
and evaluation.

The transformation is executed using DuckDB and written directly to
Parquet to avoid loading the complete dataset into pandas.

In [7]:
# ============================================================
# CELL 15 — FULL FINANCIAL & LOAN-DERIVED FEATURE TRANSFORMATION
# ============================================================

FINANCIAL_FEATURE_FILE = (
    FEATURE_OUTPUT_DIR /
    "fannie_2018Q1_financial_features.parquet"
)

print("=" * 70)
print("FULL FINANCIAL & LOAN-DERIVED FEATURE TRANSFORMATION")
print("=" * 70)

# Remove previous output if it exists
if FINANCIAL_FEATURE_FILE.exists():
    FINANCIAL_FEATURE_FILE.unlink()

financial_feature_query = f"""
COPY (
    SELECT

        -- ---------------------------------------------------
        -- Identifiers / target / time
        -- ---------------------------------------------------
        loan_id,
        reporting_date,
        target_6m,

        -- ---------------------------------------------------
        -- Original modelling variables
        -- ---------------------------------------------------
        channel,
        seller_name,
        servicer_name,
        original_interest_rate,
        current_interest_rate,
        original_upb,
        current_actual_upb,
        original_loan_term,
        origination_date,
        first_payment_date,
        loan_age,
        remaining_months_legal_maturity,
        remaining_months_maturity,
        maturity_date,
        original_ltv,
        original_cltv,
        number_of_borrowers,
        dti,
        borrower_credit_score,
        co_borrower_credit_score,
        first_time_home_buyer,
        loan_purpose,
        property_type,
        number_of_units,
        occupancy_status,
        property_state,
        msa,
        zip_code_short,
        mortgage_insurance_percentage,
        amortization_type,
        prepayment_penalty_indicator,
        interest_only_indicator,
        modification_flag,

        -- ---------------------------------------------------
        -- Payment-history features already validated
        -- ---------------------------------------------------
        ph_missing,
        ph_prior_3m_delinquency,
        ph_prior_6m_delinquency,
        ph_prior_12m_delinquency,
        ph_prior_24m_delinquency,
        ph_prior_max_delinquency,
        ph_prior_delinquency_count_6m,
        ph_prior_delinquency_count_12m,
        ph_prior_delinquency_count_24m,

        -- ---------------------------------------------------
        -- 1. Balance utilization
        -- ---------------------------------------------------
        CASE
            WHEN original_upb > 0
            THEN current_actual_upb / original_upb
            ELSE NULL
        END AS balance_utilization,

        -- ---------------------------------------------------
        -- 2. Interest-rate change
        -- ---------------------------------------------------
        CASE
            WHEN current_interest_rate IS NOT NULL
                 AND original_interest_rate IS NOT NULL
            THEN current_interest_rate - original_interest_rate
            ELSE NULL
        END AS interest_rate_change,

        -- ---------------------------------------------------
        -- 3. LTV / CLTV gap
        -- ---------------------------------------------------
        CASE
            WHEN original_ltv IS NOT NULL
                 AND original_cltv IS NOT NULL
            THEN original_cltv - original_ltv
            ELSE NULL
        END AS ltv_cltv_gap,

        -- ---------------------------------------------------
        -- 4. Loan age relative to original term
        -- ---------------------------------------------------
        CASE
            WHEN original_loan_term > 0
                 AND loan_age IS NOT NULL
            THEN loan_age / original_loan_term
            ELSE NULL
        END AS loan_age_ratio,

        -- ---------------------------------------------------
        -- 5. Remaining maturity relative to original term
        -- ---------------------------------------------------
        CASE
            WHEN original_loan_term > 0
                 AND remaining_months_maturity IS NOT NULL
            THEN remaining_months_maturity / original_loan_term
            ELSE NULL
        END AS remaining_term_ratio,

        -- ---------------------------------------------------
        -- 6. Borrower credit-score missingness
        -- ---------------------------------------------------
        CASE
            WHEN borrower_credit_score IS NULL
            THEN 1
            ELSE 0
        END AS borrower_credit_score_missing,

        -- ---------------------------------------------------
        -- 7. Co-borrower credit-score missingness
        -- ---------------------------------------------------
        CASE
            WHEN co_borrower_credit_score IS NULL
            THEN 1
            ELSE 0
        END AS co_borrower_credit_score_missing,

        -- ---------------------------------------------------
        -- 8. Mortgage-insurance missingness
        -- ---------------------------------------------------
        CASE
            WHEN mortgage_insurance_percentage IS NULL
            THEN 1
            ELSE 0
        END AS mortgage_insurance_missing,

        -- ---------------------------------------------------
        -- 9. Co-borrower / borrower score difference
        -- ---------------------------------------------------
        CASE
            WHEN borrower_credit_score IS NOT NULL
                 AND co_borrower_credit_score IS NOT NULL
            THEN co_borrower_credit_score - borrower_credit_score
            ELSE NULL
        END AS co_borrower_score_gap,

        -- ---------------------------------------------------
        -- 10–11. Reporting calendar features
        -- ---------------------------------------------------
        EXTRACT(YEAR FROM reporting_date) AS reporting_year,

        EXTRACT(MONTH FROM reporting_date) AS reporting_month

    FROM read_parquet('{PAYMENT_FEATURE_FILE}')
)
TO '{FINANCIAL_FEATURE_FILE}'
(FORMAT PARQUET, COMPRESSION ZSTD);
"""

print("Source:")
print(PAYMENT_FEATURE_FILE)

print("\nOutput:")
print(FINANCIAL_FEATURE_FILE)

print("\nStarting full transformation...")
print("Processing approximately 20.15M observations.")
print("DuckDB will write directly to Parquet.")

feature_con.execute(financial_feature_query)

print("\nTransformation completed.")
print("Output exists:", FINANCIAL_FEATURE_FILE.exists())

if FINANCIAL_FEATURE_FILE.exists():
    size_mb = FINANCIAL_FEATURE_FILE.stat().st_size / (1024 ** 2)
    print(f"Output size: {size_mb:,.2f} MB")

print("=" * 70)

FULL FINANCIAL & LOAN-DERIVED FEATURE TRANSFORMATION
Source:
C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning\data\processed\feature_engineering\fannie_2018Q1_payment_features.parquet

Output:
C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning\data\processed\feature_engineering\fannie_2018Q1_financial_features.parquet

Starting full transformation...
Processing approximately 20.15M observations.
DuckDB will write directly to Parquet.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Transformation completed.
Output exists: True
Output size: 622.23 MB


## Full Financial & Loan-Derived Feature Dataset Validation

This cell validates the completed financial-feature dataset against the
previously validated modelling dataset.

The validation confirms:

- observation count is unchanged,
- unique loan count is unchanged,
- event and non-event counts are unchanged,
- reporting-date coverage is unchanged,
- required original modelling variables are present,
- all payment-history features are present,
- all newly engineered financial features are present,
- no duplicate loan-month observations exist,
- derived feature values are finite where applicable,
- binary indicators contain only valid values,
- the financial-feature dataset has not altered the target.

No modelling or statistical fitting is performed at this stage.

In [9]:
# ============================================================
# CELL 16 — FULL FINANCIAL-FEATURE DATASET VALIDATION
# ============================================================

print("=" * 70)
print("FULL FINANCIAL-FEATURE DATASET VALIDATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. Basic dataset validation
# ------------------------------------------------------------

basic_query = f"""
SELECT
    COUNT(*) AS observations,
    COUNT(DISTINCT loan_id) AS unique_loans,

    SUM(CASE WHEN target_6m = 1 THEN 1 ELSE 0 END)
        AS events,

    SUM(CASE WHEN target_6m = 0 THEN 1 ELSE 0 END)
        AS non_events,

    COUNT(DISTINCT reporting_date)
        AS reporting_months,

    MIN(reporting_date)
        AS min_reporting_date,

    MAX(reporting_date)
        AS max_reporting_date,

    SUM(CASE WHEN loan_id IS NULL THEN 1 ELSE 0 END)
        AS null_loan_id,

    SUM(CASE WHEN reporting_date IS NULL THEN 1 ELSE 0 END)
        AS null_reporting_date,

    SUM(CASE WHEN target_6m IS NULL THEN 1 ELSE 0 END)
        AS null_target

FROM read_parquet('{FINANCIAL_FEATURE_FILE}')
"""

basic_df = feature_con.execute(
    basic_query
).fetchdf()

display(basic_df)

# ------------------------------------------------------------
# 2. Validate required columns
# ------------------------------------------------------------

schema_df = feature_con.execute(
    f"""
    DESCRIBE
    SELECT *
    FROM read_parquet('{FINANCIAL_FEATURE_FILE}')
    """
).fetchdf()

# DuckDB DESCRIBE uses 'column_name'
available_columns = set(
    schema_df["column_name"].tolist()
)

required_columns = [
    # Core
    "loan_id",
    "reporting_date",
    "target_6m",

    # Original financial / loan variables
    "original_interest_rate",
    "current_interest_rate",
    "original_upb",
    "current_actual_upb",
    "original_loan_term",
    "origination_date",
    "first_payment_date",
    "loan_age",
    "remaining_months_legal_maturity",
    "remaining_months_maturity",
    "maturity_date",
    "original_ltv",
    "original_cltv",
    "number_of_borrowers",
    "dti",
    "borrower_credit_score",
    "co_borrower_credit_score",
    "first_time_home_buyer",
    "loan_purpose",
    "property_type",
    "number_of_units",
    "occupancy_status",
    "property_state",
    "msa",
    "zip_code_short",
    "mortgage_insurance_percentage",
    "amortization_type",
    "prepayment_penalty_indicator",
    "interest_only_indicator",
    "modification_flag",

    # Payment-history features
    "ph_missing",
    "ph_prior_3m_delinquency",
    "ph_prior_6m_delinquency",
    "ph_prior_12m_delinquency",
    "ph_prior_24m_delinquency",
    "ph_prior_max_delinquency",
    "ph_prior_delinquency_count_6m",
    "ph_prior_delinquency_count_12m",
    "ph_prior_delinquency_count_24m",

    # Financial-derived features
    "balance_utilization",
    "interest_rate_change",
    "ltv_cltv_gap",
    "loan_age_ratio",
    "remaining_term_ratio",
    "borrower_credit_score_missing",
    "co_borrower_credit_score_missing",
    "mortgage_insurance_missing",
    "co_borrower_score_gap",
    "reporting_year",
    "reporting_month"
]

missing_required_columns = [
    col
    for col in required_columns
    if col not in available_columns
]

print("\nTotal output columns:", len(available_columns))
print("Required columns checked:", len(required_columns))
print("Missing required columns:", len(missing_required_columns))

if missing_required_columns:
    print("\nMissing columns:")
    for col in missing_required_columns:
        print(" -", col)
else:
    print("All required columns are present.")

# ------------------------------------------------------------
# 3. Duplicate loan-month check
# ------------------------------------------------------------

duplicate_query = f"""
SELECT COUNT(*) AS duplicate_loan_month_groups
FROM (
    SELECT
        loan_id,
        reporting_date,
        COUNT(*) AS n
    FROM read_parquet('{FINANCIAL_FEATURE_FILE}')
    GROUP BY loan_id, reporting_date
    HAVING COUNT(*) > 1
)
"""

duplicate_df = feature_con.execute(
    duplicate_query
).fetchdf()

duplicate_groups = int(
    duplicate_df.loc[0, "duplicate_loan_month_groups"]
)

print("\nDuplicate loan-month groups:", duplicate_groups)

# ------------------------------------------------------------
# 4. Target consistency
# ------------------------------------------------------------

target_check_query = f"""
SELECT
    COUNT(*) AS observations,

    SUM(
        CASE WHEN target_6m = 1
        THEN 1 ELSE 0 END
    ) AS events,

    SUM(
        CASE WHEN target_6m = 0
        THEN 1 ELSE 0 END
    ) AS non_events

FROM read_parquet('{FINANCIAL_FEATURE_FILE}')
"""

target_check_df = feature_con.execute(
    target_check_query
).fetchdf()

target_observations = int(
    target_check_df.loc[0, "observations"]
)

target_events = int(
    target_check_df.loc[0, "events"]
)

target_non_events = int(
    target_check_df.loc[0, "non_events"]
)

# ------------------------------------------------------------
# 5. Derived-feature diagnostics
# ------------------------------------------------------------

derived_query = f"""
SELECT

    -- Balance utilization
    MIN(balance_utilization)
        AS min_balance_utilization,

    MAX(balance_utilization)
        AS max_balance_utilization,

    SUM(
        CASE
            WHEN balance_utilization IS NOT NULL
                 AND NOT isfinite(balance_utilization)
            THEN 1
            ELSE 0
        END
    ) AS nonfinite_balance_utilization,

    -- Interest-rate change
    MIN(interest_rate_change)
        AS min_interest_rate_change,

    MAX(interest_rate_change)
        AS max_interest_rate_change,

    -- LTV / CLTV gap
    MIN(ltv_cltv_gap)
        AS min_ltv_cltv_gap,

    MAX(ltv_cltv_gap)
        AS max_ltv_cltv_gap,

    -- Loan age ratio
    MIN(loan_age_ratio)
        AS min_loan_age_ratio,

    MAX(loan_age_ratio)
        AS max_loan_age_ratio,

    -- Remaining term ratio
    MIN(remaining_term_ratio)
        AS min_remaining_term_ratio,

    MAX(remaining_term_ratio)
        AS max_remaining_term_ratio,

    -- Co-borrower score gap
    MIN(co_borrower_score_gap)
        AS min_score_gap,

    MAX(co_borrower_score_gap)
        AS max_score_gap,

    -- Binary indicators
    MIN(ph_missing) AS min_ph_missing,
    MAX(ph_missing) AS max_ph_missing,

    MIN(borrower_credit_score_missing)
        AS min_borrower_score_missing,

    MAX(borrower_credit_score_missing)
        AS max_borrower_score_missing,

    MIN(co_borrower_credit_score_missing)
        AS min_co_borrower_score_missing,

    MAX(co_borrower_credit_score_missing)
        AS max_co_borrower_score_missing,

    MIN(mortgage_insurance_missing)
        AS min_mortgage_insurance_missing,

    MAX(mortgage_insurance_missing)
        AS max_mortgage_insurance_missing,

    -- Calendar features
    MIN(reporting_month)
        AS min_reporting_month,

    MAX(reporting_month)
        AS max_reporting_month,

    MIN(reporting_year)
        AS min_reporting_year,

    MAX(reporting_year)
        AS max_reporting_year

FROM read_parquet('{FINANCIAL_FEATURE_FILE}')
"""

derived_df = feature_con.execute(
    derived_query
).fetchdf()

print("\nDerived-feature diagnostics:")
display(derived_df)

# ------------------------------------------------------------
# 6. NULL counts in engineered variables
# ------------------------------------------------------------

null_query = f"""
SELECT

    SUM(
        CASE WHEN balance_utilization IS NULL
        THEN 1 ELSE 0 END
    ) AS null_balance_utilization,

    SUM(
        CASE WHEN interest_rate_change IS NULL
        THEN 1 ELSE 0 END
    ) AS null_interest_rate_change,

    SUM(
        CASE WHEN ltv_cltv_gap IS NULL
        THEN 1 ELSE 0 END
    ) AS null_ltv_cltv_gap,

    SUM(
        CASE WHEN loan_age_ratio IS NULL
        THEN 1 ELSE 0 END
    ) AS null_loan_age_ratio,

    SUM(
        CASE WHEN remaining_term_ratio IS NULL
        THEN 1 ELSE 0 END
    ) AS null_remaining_term_ratio,

    SUM(
        CASE WHEN ph_prior_delinquency_count_6m IS NULL
        THEN 1 ELSE 0 END
    ) AS null_count_6m,

    SUM(
        CASE WHEN ph_prior_delinquency_count_12m IS NULL
        THEN 1 ELSE 0 END
    ) AS null_count_12m,

    SUM(
        CASE WHEN ph_prior_delinquency_count_24m IS NULL
        THEN 1 ELSE 0 END
    ) AS null_count_24m

FROM read_parquet('{FINANCIAL_FEATURE_FILE}')
"""

null_df = feature_con.execute(
    null_query
).fetchdf()

print("\nNULL counts in engineered variables:")
display(null_df)

# ------------------------------------------------------------
# 7. Final PASS / FAIL checks
# ------------------------------------------------------------

observations_ok = (
    target_observations == 20_155_269
)

loans_ok = (
    int(basic_df.loc[0, "unique_loans"])
    == 450_106
)

events_ok = (
    target_events == 646_957
)

non_events_ok = (
    target_non_events == 19_508_312
)

dates_ok = (
    str(basic_df.loc[0, "min_reporting_date"])[:10]
    == "2018-01-01"
    and
    str(basic_df.loc[0, "max_reporting_date"])[:10]
    == "2025-09-01"
)

null_core_ok = (
    int(basic_df.loc[0, "null_loan_id"]) == 0
    and
    int(basic_df.loc[0, "null_reporting_date"]) == 0
    and
    int(basic_df.loc[0, "null_target"]) == 0
)

columns_ok = (
    len(missing_required_columns) == 0
)

duplicates_ok = (
    duplicate_groups == 0
)

balance_finite_ok = (
    int(
        derived_df.loc[
            0,
            "nonfinite_balance_utilization"
        ]
    ) == 0
)

calendar_ok = (
    1 <= int(
        derived_df.loc[0, "min_reporting_month"]
    ) <= 12
    and
    1 <= int(
        derived_df.loc[0, "max_reporting_month"]
    ) <= 12
)

binary_ok = (
    int(derived_df.loc[0, "min_ph_missing"]) >= 0
    and
    int(derived_df.loc[0, "max_ph_missing"]) <= 1

    and
    int(
        derived_df.loc[
            0,
            "min_borrower_score_missing"
        ]
    ) >= 0

    and
    int(
        derived_df.loc[
            0,
            "max_borrower_score_missing"
        ]
    ) <= 1

    and
    int(
        derived_df.loc[
            0,
            "min_co_borrower_score_missing"
        ]
    ) >= 0

    and
    int(
        derived_df.loc[
            0,
            "max_co_borrower_score_missing"
        ]
    ) <= 1

    and
    int(
        derived_df.loc[
            0,
            "min_mortgage_insurance_missing"
        ]
    ) >= 0

    and
    int(
        derived_df.loc[
            0,
            "max_mortgage_insurance_missing"
        ]
    ) <= 1
)

# ------------------------------------------------------------
# 8. Print final validation result
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("VALIDATION RESULT")
print("=" * 70)

checks = {
    "Observation count": observations_ok,
    "Unique loan count": loans_ok,
    "Event count": events_ok,
    "Non-event count": non_events_ok,
    "Reporting-date range": dates_ok,
    "No NULL core fields": null_core_ok,
    "Required columns": columns_ok,
    "No duplicate loan-months": duplicates_ok,
    "Balance utilization finite": balance_finite_ok,
    "Reporting month valid": calendar_ok,
    "Binary indicators valid": binary_ok
}

for name, passed in checks.items():
    print(
        f"{name:<32}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

overall_pass = all(checks.values())

print("\nOVERALL:", "PASS" if overall_pass else "FAIL")
print("=" * 70)

FULL FINANCIAL-FEATURE DATASET VALIDATION


,observations,unique_loans,events,non_events,reporting_months,min_reporting_date,max_reporting_date,null_loan_id,null_reporting_date,null_target
0,20155269,450106,646957.0,19508312.0,93,2018-01-01,2025-09-01,0.0,0.0,0.0



Total output columns: 56
Required columns checked: 53
Missing required columns: 0
All required columns are present.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Duplicate loan-month groups: 0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Derived-feature diagnostics:


,min_balance_utilization,max_balance_utilization,nonfinite_balance_utilization,min_interest_rate_change,max_interest_rate_change,min_ltv_cltv_gap,max_ltv_cltv_gap,min_loan_age_ratio,max_loan_age_ratio,min_remaining_term_ratio,...,min_borrower_score_missing,max_borrower_score_missing,min_co_borrower_score_missing,max_co_borrower_score_missing,min_mortgage_insurance_missing,max_mortgage_insurance_missing,min_reporting_month,max_reporting_month,min_reporting_year,max_reporting_year
0,0.0,1.784416,0.0,-3.0,1.0,0.0,78.0,-0.013889,0.947917,0.002778,...,0,1,0,1,0,1,1,12,2018,2025



NULL counts in engineered variables:


,null_balance_utilization,null_interest_rate_change,null_ltv_cltv_gap,null_loan_age_ratio,null_remaining_term_ratio,null_count_6m,null_count_12m,null_count_24m
0,0.0,0.0,26.0,0.0,142201.0,10315042.0,10315042.0,10315042.0



VALIDATION RESULT
Observation count               : PASS
Unique loan count               : PASS
Event count                     : PASS
Non-event count                 : PASS
Reporting-date range            : PASS
No NULL core fields             : PASS
Required columns                : PASS
No duplicate loan-months        : PASS
Balance utilization finite      : PASS
Reporting month valid           : PASS
Binary indicators valid         : PASS

OVERALL: PASS


## Temporal / Dynamic Feature Engineering — Corrected Formula Test

The previous test sampled individual observations before calculating
monthly lags. This can remove the preceding observation of a loan and
therefore cannot properly validate month-over-month features.

This corrected test first selects a small number of complete loans and
then retrieves their chronological observations.

The temporal features are calculated using the immediately preceding
observation within each loan.

A previous observation is considered valid only when its reporting date
is exactly one calendar month before the current reporting date.

This ensures that reporting gaps are not incorrectly treated as
month-over-month changes.

The following dynamic features are tested:

1. prior_month_balance
2. balance_change
3. balance_change_pct
4. prior_month_interest_rate
5. interest_rate_change_1m
6. prior_month_loan_age
7. loan_age_change
8. prior_month_remaining_maturity
9. remaining_maturity_change

No target information is used.

In [11]:
# ============================================================
# CELL 17 — CORRECTED TEMPORAL / DYNAMIC FEATURE TEST
# ============================================================

print("=" * 70)
print("CORRECTED TEMPORAL / DYNAMIC FEATURE TEST")
print("=" * 70)

temporal_test_query = f"""
WITH selected_loans AS (

    -- Select complete loan histories rather than
    -- randomly sampling individual observations.
    SELECT loan_id
    FROM read_parquet('{FINANCIAL_FEATURE_FILE}')
    GROUP BY loan_id
    HAVING COUNT(*) >= 6
    ORDER BY loan_id
    LIMIT 20
),

loan_history AS (

    SELECT
        f.*
    FROM read_parquet('{FINANCIAL_FEATURE_FILE}') f
    INNER JOIN selected_loans s
        ON f.loan_id = s.loan_id
),

lagged AS (

    SELECT

        loan_id,
        reporting_date,

        current_actual_upb,
        current_interest_rate,
        loan_age,
        remaining_months_maturity,

        LAG(current_actual_upb)
            OVER (
                PARTITION BY loan_id
                ORDER BY reporting_date
            ) AS prior_month_balance,

        LAG(current_interest_rate)
            OVER (
                PARTITION BY loan_id
                ORDER BY reporting_date
            ) AS prior_month_interest_rate,

        LAG(loan_age)
            OVER (
                PARTITION BY loan_id
                ORDER BY reporting_date
            ) AS prior_month_loan_age,

        LAG(remaining_months_maturity)
            OVER (
                PARTITION BY loan_id
                ORDER BY reporting_date
            ) AS prior_month_remaining_maturity,

        LAG(reporting_date)
            OVER (
                PARTITION BY loan_id
                ORDER BY reporting_date
            ) AS prior_reporting_date

    FROM loan_history
)

SELECT

    loan_id,
    reporting_date,
    prior_reporting_date,

    current_actual_upb,
    prior_month_balance,

    current_interest_rate,
    prior_month_interest_rate,

    loan_age,
    prior_month_loan_age,

    remaining_months_maturity,
    prior_month_remaining_maturity,

    -- -------------------------------------------------------
    -- Calendar continuity
    -- -------------------------------------------------------

    CASE
        WHEN prior_reporting_date IS NOT NULL
             AND prior_reporting_date =
                 reporting_date - INTERVAL '1 month'
        THEN 1
        ELSE 0
    END AS has_valid_prior_month,

    -- -------------------------------------------------------
    -- Balance change
    -- -------------------------------------------------------

    CASE
        WHEN prior_reporting_date =
             reporting_date - INTERVAL '1 month'
             AND prior_month_balance IS NOT NULL
        THEN current_actual_upb - prior_month_balance
        ELSE NULL
    END AS balance_change,

    -- -------------------------------------------------------
    -- Balance percentage change
    -- -------------------------------------------------------

    CASE
        WHEN prior_reporting_date =
             reporting_date - INTERVAL '1 month'
             AND prior_month_balance > 0
        THEN
            (current_actual_upb - prior_month_balance)
            / prior_month_balance
        ELSE NULL
    END AS balance_change_pct,

    -- -------------------------------------------------------
    -- One-month interest-rate change
    -- -------------------------------------------------------

    CASE
        WHEN prior_reporting_date =
             reporting_date - INTERVAL '1 month'
             AND current_interest_rate IS NOT NULL
             AND prior_month_interest_rate IS NOT NULL
        THEN current_interest_rate - prior_month_interest_rate
        ELSE NULL
    END AS interest_rate_change_1m,

    -- -------------------------------------------------------
    -- Loan-age change
    -- -------------------------------------------------------

    CASE
        WHEN prior_reporting_date =
             reporting_date - INTERVAL '1 month'
             AND loan_age IS NOT NULL
             AND prior_month_loan_age IS NOT NULL
        THEN loan_age - prior_month_loan_age
        ELSE NULL
    END AS loan_age_change,

    -- -------------------------------------------------------
    -- Remaining-maturity change
    -- -------------------------------------------------------

    CASE
        WHEN prior_reporting_date =
             reporting_date - INTERVAL '1 month'
             AND remaining_months_maturity IS NOT NULL
             AND prior_month_remaining_maturity IS NOT NULL
        THEN
            remaining_months_maturity
            - prior_month_remaining_maturity
        ELSE NULL
    END AS remaining_maturity_change

FROM lagged

ORDER BY loan_id, reporting_date
"""

temporal_test_df = feature_con.execute(
    temporal_test_query
).fetchdf()

print(
    f"Test observations: "
    f"{len(temporal_test_df):,}"
)

print(
    f"Test loans: "
    f"{temporal_test_df['loan_id'].nunique():,}"
)

print("\nTemporal sample:")
display(temporal_test_df.head(25))

# ------------------------------------------------------------
# Valid prior-month observations
# ------------------------------------------------------------

print("\nValid prior-month observations:")

prior_counts = (
    temporal_test_df[
        "has_valid_prior_month"
    ]
    .value_counts()
    .sort_index()
)

print(prior_counts)

# ------------------------------------------------------------
# Dynamic-feature summary
# ------------------------------------------------------------

print("\nDynamic-feature summary:")

display(
    temporal_test_df[
        [
            "balance_change",
            "balance_change_pct",
            "interest_rate_change_1m",
            "loan_age_change",
            "remaining_maturity_change"
        ]
    ].describe()
)

# ------------------------------------------------------------
# Explicit continuity audit
# ------------------------------------------------------------

continuity_check = temporal_test_df[
    temporal_test_df["has_valid_prior_month"] == 1
].copy()

if len(continuity_check) > 0:

    continuity_check["expected_prior_date"] = (
        continuity_check["reporting_date"]
        - pd.DateOffset(months=1)
    )

    continuity_mismatches = (
        continuity_check[
            continuity_check["prior_reporting_date"]
            != continuity_check["expected_prior_date"]
        ]
    )

else:

    continuity_mismatches = continuity_check

print("\nCalendar-continuity mismatches:")
print(len(continuity_mismatches))

# ------------------------------------------------------------
# Formula checks
# ------------------------------------------------------------

print("\nFormula checks:")

checks = {

    "Sample is non-empty":
        len(temporal_test_df) > 0,

    "At least one valid prior month":
        (
            temporal_test_df[
                "has_valid_prior_month"
            ].sum() > 0
        ),

    "Prior-month flag is binary":
        set(
            temporal_test_df[
                "has_valid_prior_month"
            ].dropna().unique()
        ).issubset({0, 1}),

    "Calendar continuity correct":
        len(continuity_mismatches) == 0,

    "Balance change finite":
        np.isfinite(
            temporal_test_df[
                "balance_change"
            ].dropna()
        ).all(),

    "Balance change pct finite":
        np.isfinite(
            temporal_test_df[
                "balance_change_pct"
            ].dropna()
        ).all(),

    "Interest-rate change finite":
        np.isfinite(
            temporal_test_df[
                "interest_rate_change_1m"
            ].dropna()
        ).all(),

    "Loan-age change finite":
        np.isfinite(
            temporal_test_df[
                "loan_age_change"
            ].dropna()
        ).all(),

    "Maturity change finite":
        np.isfinite(
            temporal_test_df[
                "remaining_maturity_change"
            ].dropna()
        ).all()
}

for name, passed in checks.items():

    print(
        f"{name:<38}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

overall_pass = all(checks.values())

print("\n" + "=" * 70)
print(
    "OVERALL:",
    "PASS" if overall_pass else "FAIL"
)
print("=" * 70)

CORRECTED TEMPORAL / DYNAMIC FEATURE TEST


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Test observations: 799
Test loans: 20

Temporal sample:


,loan_id,reporting_date,prior_reporting_date,current_actual_upb,prior_month_balance,current_interest_rate,prior_month_interest_rate,loan_age,prior_month_loan_age,remaining_months_maturity,prior_month_remaining_maturity,has_valid_prior_month,balance_change,balance_change_pct,interest_rate_change_1m,loan_age_change,remaining_maturity_change
0,100001040173,2018-02-01,NaT,0.00,NaN,4.25,NaN,0.0,NaN,360.0,NaN,0,NaN,NaN,NaN,NaN,NaN
1,100001040173,2018-03-01,2018-02-01,0.00,0.00,4.25,4.25,1.0,0.0,359.0,360.0,1,0.00,NaN,0.0,1.0,-1.0
2,100001040173,2018-04-01,2018-03-01,0.00,0.00,4.25,4.25,2.0,1.0,358.0,359.0,1,0.00,NaN,0.0,1.0,-1.0
3,100001040173,2018-05-01,2018-04-01,0.00,0.00,4.25,4.25,3.0,2.0,357.0,358.0,1,0.00,NaN,0.0,1.0,-1.0
4,100001040173,2018-06-01,2018-05-01,0.00,0.00,4.25,4.25,4.0,3.0,356.0,357.0,1,0.00,NaN,0.0,1.0,-1.0
5,100001040173,2018-07-01,2018-06-01,0.00,0.00,4.25,4.25,5.0,4.0,355.0,356.0,1,0.00,NaN,0.0,1.0,-1.0
6,100001040173,2018-08-01,2018-07-01,449321.18,0.00,4.25,4.25,6.0,5.0,354.0,355.0,1,449321.18,NaN,0.0,1.0,-1.0
7,100001040173,2018-09-01,2018-08-01,448683.55,449321.18,4.25,4.25,7.0,6.0,353.0,354.0,1,-637.63,-0.001419,0.0,1.0,-1.0
8,100001040173,2018-10-01,2018-09-01,448043.66,448683.55,4.25,4.25,8.0,7.0,352.0,353.0,1,-639.89,-0.001426,0.0,1.0,-1.0
9,100001040173,2018-11-01,2018-10-01,447401.50,448043.66,4.25,4.25,9.0,8.0,351.0,352.0,1,-642.16,-0.001433,0.0,1.0,-1.0



Valid prior-month observations:
has_valid_prior_month
0     25
1    774
Name: count, dtype: int64

Dynamic-feature summary:


,balance_change,balance_change_pct,interest_rate_change_1m,loan_age_change,remaining_maturity_change
count,774.000000,654.000000,774.0,774.0,774.000000
mean,5395.182261,-0.001772,0.0,1.0,-1.006460
std,39454.406089,0.001000,0.0,0.0,0.497039
min,-1918.390000,-0.007373,0.0,1.0,-5.000000
25%,-337.322500,-0.001865,0.0,1.0,-1.000000
50%,-236.195000,-0.001581,0.0,1.0,-1.000000
75%,-127.700000,-0.001445,0.0,1.0,-1.000000
max,540752.220000,0.000000,0.0,1.0,0.000000



Calendar-continuity mismatches:
0

Formula checks:
Sample is non-empty                   : PASS
At least one valid prior month        : PASS
Prior-month flag is binary            : PASS
Calendar continuity correct           : PASS
Balance change finite                 : PASS
Balance change pct finite             : PASS
Interest-rate change finite           : PASS
Loan-age change finite                : PASS
Maturity change finite                : PASS

OVERALL: PASS


## Full Temporal / Dynamic Feature Transformation

The validated month-over-month temporal features are now generated for the
complete financial-feature dataset.

Temporal features are calculated within each loan and ordered by
reporting_date.

A lag is considered valid only when the preceding observation belongs to
the immediately preceding calendar month. This prevents reporting gaps
from being incorrectly interpreted as normal month-over-month changes.

The following features are added:

- prior_month_balance
- balance_change
- balance_change_pct
- prior_month_interest_rate
- interest_rate_change_1m
- prior_month_loan_age
- loan_age_change
- prior_month_remaining_maturity
- remaining_maturity_change
- has_valid_prior_month

No target information is used in the construction of these variables.

The complete dataset is written directly to Parquet using DuckDB with
disk spilling enabled.

In [12]:
# ============================================================
# CELL 18 — FULL TEMPORAL / DYNAMIC FEATURE TRANSFORMATION
# ============================================================

TEMPORAL_FEATURE_FILE = (
    FEATURE_OUTPUT_DIR /
    "fannie_2018Q1_temporal_features.parquet"
)

print("=" * 70)
print("FULL TEMPORAL / DYNAMIC FEATURE TRANSFORMATION")
print("=" * 70)

print("\nSource:")
print(FINANCIAL_FEATURE_FILE)

print("\nOutput:")
print(TEMPORAL_FEATURE_FILE)

# ------------------------------------------------------------
# Remove previous output if it exists
# ------------------------------------------------------------

if TEMPORAL_FEATURE_FILE.exists():
    TEMPORAL_FEATURE_FILE.unlink()

# ------------------------------------------------------------
# Full temporal transformation
# ------------------------------------------------------------

temporal_feature_query = f"""
COPY (

    WITH lagged AS (

        SELECT
            *,

            -- Previous observation within the same loan
            LAG(current_actual_upb)
                OVER (
                    PARTITION BY loan_id
                    ORDER BY reporting_date
                ) AS prior_month_balance,

            LAG(current_interest_rate)
                OVER (
                    PARTITION BY loan_id
                    ORDER BY reporting_date
                ) AS prior_month_interest_rate,

            LAG(loan_age)
                OVER (
                    PARTITION BY loan_id
                    ORDER BY reporting_date
                ) AS prior_month_loan_age,

            LAG(remaining_months_maturity)
                OVER (
                    PARTITION BY loan_id
                    ORDER BY reporting_date
                ) AS prior_month_remaining_maturity,

            LAG(reporting_date)
                OVER (
                    PARTITION BY loan_id
                    ORDER BY reporting_date
                ) AS prior_reporting_date

        FROM read_parquet(
            '{FINANCIAL_FEATURE_FILE}'
        )
    )

    SELECT

        * EXCLUDE (
            prior_reporting_date,
            prior_month_balance,
            prior_month_interest_rate,
            prior_month_loan_age,
            prior_month_remaining_maturity
        ),

        -- ----------------------------------------------------
        -- Valid previous-calendar-month indicator
        -- ----------------------------------------------------

        CASE
            WHEN prior_reporting_date IS NOT NULL
                 AND prior_reporting_date =
                     reporting_date - INTERVAL '1 month'
            THEN 1
            ELSE 0
        END AS has_valid_prior_month,

        -- ----------------------------------------------------
        -- Previous-month balance
        -- ----------------------------------------------------

        CASE
            WHEN prior_reporting_date =
                 reporting_date - INTERVAL '1 month'
            THEN prior_month_balance
            ELSE NULL
        END AS prior_month_balance,

        -- ----------------------------------------------------
        -- Balance change
        -- ----------------------------------------------------

        CASE
            WHEN prior_reporting_date =
                 reporting_date - INTERVAL '1 month'
                 AND prior_month_balance IS NOT NULL
            THEN current_actual_upb - prior_month_balance
            ELSE NULL
        END AS balance_change,

        -- ----------------------------------------------------
        -- Balance percentage change
        -- ----------------------------------------------------

        CASE
            WHEN prior_reporting_date =
                 reporting_date - INTERVAL '1 month'
                 AND prior_month_balance > 0
            THEN
                (current_actual_upb - prior_month_balance)
                / prior_month_balance
            ELSE NULL
        END AS balance_change_pct,

        -- ----------------------------------------------------
        -- Previous-month interest rate
        -- ----------------------------------------------------

        CASE
            WHEN prior_reporting_date =
                 reporting_date - INTERVAL '1 month'
            THEN prior_month_interest_rate
            ELSE NULL
        END AS prior_month_interest_rate,

        -- ----------------------------------------------------
        -- One-month interest-rate change
        -- ----------------------------------------------------

        CASE
            WHEN prior_reporting_date =
                 reporting_date - INTERVAL '1 month'
                 AND current_interest_rate IS NOT NULL
                 AND prior_month_interest_rate IS NOT NULL
            THEN
                current_interest_rate
                - prior_month_interest_rate
            ELSE NULL
        END AS interest_rate_change_1m,

        -- ----------------------------------------------------
        -- Previous-month loan age
        -- ----------------------------------------------------

        CASE
            WHEN prior_reporting_date =
                 reporting_date - INTERVAL '1 month'
            THEN prior_month_loan_age
            ELSE NULL
        END AS prior_month_loan_age,

        -- ----------------------------------------------------
        -- Loan-age change
        -- ----------------------------------------------------

        CASE
            WHEN prior_reporting_date =
                 reporting_date - INTERVAL '1 month'
                 AND loan_age IS NOT NULL
                 AND prior_month_loan_age IS NOT NULL
            THEN
                loan_age - prior_month_loan_age
            ELSE NULL
        END AS loan_age_change,

        -- ----------------------------------------------------
        -- Previous-month remaining maturity
        -- ----------------------------------------------------

        CASE
            WHEN prior_reporting_date =
                 reporting_date - INTERVAL '1 month'
            THEN prior_month_remaining_maturity
            ELSE NULL
        END AS prior_month_remaining_maturity,

        -- ----------------------------------------------------
        -- Remaining-maturity change
        -- ----------------------------------------------------

        CASE
            WHEN prior_reporting_date =
                 reporting_date - INTERVAL '1 month'
                 AND remaining_months_maturity IS NOT NULL
                 AND prior_month_remaining_maturity IS NOT NULL
            THEN
                remaining_months_maturity
                - prior_month_remaining_maturity
            ELSE NULL
        END AS remaining_maturity_change

    FROM lagged

)
TO '{TEMPORAL_FEATURE_FILE}'
(FORMAT PARQUET, COMPRESSION ZSTD);
"""

print("\nStarting full transformation...")
print("Approximately 20.15M observations will be processed.")
print("DuckDB window functions may require disk spilling.")
print("Please allow the cell to complete.")

feature_con.execute(
    temporal_feature_query
)

print("\nTransformation completed.")
print(
    "Output exists:",
    TEMPORAL_FEATURE_FILE.exists()
)

if TEMPORAL_FEATURE_FILE.exists():

    size_mb = (
        TEMPORAL_FEATURE_FILE.stat().st_size
        / (1024 ** 2)
    )

    print(
        f"Output size: {size_mb:,.2f} MB"
    )

print("=" * 70)

FULL TEMPORAL / DYNAMIC FEATURE TRANSFORMATION

Source:
C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning\data\processed\feature_engineering\fannie_2018Q1_financial_features.parquet

Output:
C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning\data\processed\feature_engineering\fannie_2018Q1_temporal_features.parquet

Starting full transformation...
Approximately 20.15M observations will be processed.
DuckDB window functions may require disk spilling.
Please allow the cell to complete.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Transformation completed.
Output exists: True
Output size: 520.21 MB


## Full Temporal / Dynamic Feature Dataset Validation

This cell validates the completed temporal-feature dataset against the
previously validated financial-feature dataset.

The validation confirms:

- observation count is unchanged,
- unique loan count is unchanged,
- event and non-event counts are unchanged,
- reporting-date coverage is unchanged,
- no duplicate loan-month observations exist,
- the valid-prior-month indicator is binary,
- temporal features are finite where populated,
- temporal features are NULL when a valid preceding calendar month does
  not exist,
- month-over-month loan-age changes are internally consistent.

No target-derived information is used.

In [13]:
# ============================================================
# CELL 19 — FULL TEMPORAL FEATURE DATASET VALIDATION
# ============================================================

print("=" * 70)
print("FULL TEMPORAL / DYNAMIC FEATURE DATASET VALIDATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. Basic dataset validation
# ------------------------------------------------------------

basic_query = f"""
SELECT
    COUNT(*) AS observations,
    COUNT(DISTINCT loan_id) AS unique_loans,

    SUM(
        CASE WHEN target_6m = 1
        THEN 1 ELSE 0 END
    ) AS events,

    SUM(
        CASE WHEN target_6m = 0
        THEN 1 ELSE 0 END
    ) AS non_events,

    COUNT(DISTINCT reporting_date)
        AS reporting_months,

    MIN(reporting_date)
        AS min_reporting_date,

    MAX(reporting_date)
        AS max_reporting_date,

    SUM(
        CASE WHEN loan_id IS NULL
        THEN 1 ELSE 0 END
    ) AS null_loan_id,

    SUM(
        CASE WHEN reporting_date IS NULL
        THEN 1 ELSE 0 END
    ) AS null_reporting_date,

    SUM(
        CASE WHEN target_6m IS NULL
        THEN 1 ELSE 0 END
    ) AS null_target

FROM read_parquet('{TEMPORAL_FEATURE_FILE}')
"""

basic_df = feature_con.execute(
    basic_query
).fetchdf()

display(basic_df)

# ------------------------------------------------------------
# 2. Validate required temporal columns
# ------------------------------------------------------------

schema_df = feature_con.execute(
    f"""
    DESCRIBE
    SELECT *
    FROM read_parquet('{TEMPORAL_FEATURE_FILE}')
    """
).fetchdf()

available_columns = set(
    schema_df["column_name"].tolist()
)

required_temporal_columns = [
    "loan_id",
    "reporting_date",
    "target_6m",

    "prior_month_balance",
    "balance_change",
    "balance_change_pct",

    "prior_month_interest_rate",
    "interest_rate_change_1m",

    "prior_month_loan_age",
    "loan_age_change",

    "prior_month_remaining_maturity",
    "remaining_maturity_change",

    "has_valid_prior_month"
]

missing_temporal_columns = [
    col
    for col in required_temporal_columns
    if col not in available_columns
]

print("\nTotal output columns:", len(available_columns))
print(
    "Required temporal columns checked:",
    len(required_temporal_columns)
)

print(
    "Missing temporal columns:",
    len(missing_temporal_columns)
)

if missing_temporal_columns:
    print("\nMissing columns:")
    for col in missing_temporal_columns:
        print(" -", col)
else:
    print("All required temporal columns are present.")

# ------------------------------------------------------------
# 3. Duplicate loan-month check
# ------------------------------------------------------------

duplicate_query = f"""
SELECT COUNT(*) AS duplicate_loan_month_groups
FROM (
    SELECT
        loan_id,
        reporting_date,
        COUNT(*) AS n
    FROM read_parquet('{TEMPORAL_FEATURE_FILE}')
    GROUP BY loan_id, reporting_date
    HAVING COUNT(*) > 1
)
"""

duplicate_df = feature_con.execute(
    duplicate_query
).fetchdf()

duplicate_groups = int(
    duplicate_df.loc[
        0,
        "duplicate_loan_month_groups"
    ]
)

print(
    "\nDuplicate loan-month groups:",
    duplicate_groups
)

# ------------------------------------------------------------
# 4. Temporal feature diagnostics
# ------------------------------------------------------------

temporal_diag_query = f"""
SELECT

    -- Prior-month validity
    MIN(has_valid_prior_month)
        AS min_prior_flag,

    MAX(has_valid_prior_month)
        AS max_prior_flag,

    SUM(
        CASE
            WHEN has_valid_prior_month = 1
            THEN 1
            ELSE 0
        END
    ) AS valid_prior_month_rows,

    SUM(
        CASE
            WHEN has_valid_prior_month = 0
            THEN 1
            ELSE 0
        END
    ) AS invalid_prior_month_rows,

    -- Balance change
    MIN(balance_change)
        AS min_balance_change,

    MAX(balance_change)
        AS max_balance_change,

    SUM(
        CASE
            WHEN balance_change IS NOT NULL
                 AND NOT isfinite(balance_change)
            THEN 1
            ELSE 0
        END
    ) AS nonfinite_balance_change,

    -- Balance percentage change
    MIN(balance_change_pct)
        AS min_balance_change_pct,

    MAX(balance_change_pct)
        AS max_balance_change_pct,

    SUM(
        CASE
            WHEN balance_change_pct IS NOT NULL
                 AND NOT isfinite(balance_change_pct)
            THEN 1
            ELSE 0
        END
    ) AS nonfinite_balance_change_pct,

    -- Interest-rate change
    MIN(interest_rate_change_1m)
        AS min_interest_rate_change_1m,

    MAX(interest_rate_change_1m)
        AS max_interest_rate_change_1m,

    SUM(
        CASE
            WHEN interest_rate_change_1m IS NOT NULL
                 AND NOT isfinite(interest_rate_change_1m)
            THEN 1
            ELSE 0
        END
    ) AS nonfinite_rate_change,

    -- Loan-age change
    MIN(loan_age_change)
        AS min_loan_age_change,

    MAX(loan_age_change)
        AS max_loan_age_change,

    SUM(
        CASE
            WHEN loan_age_change IS NOT NULL
                 AND NOT isfinite(loan_age_change)
            THEN 1
            ELSE 0
        END
    ) AS nonfinite_age_change,

    -- Remaining maturity change
    MIN(remaining_maturity_change)
        AS min_remaining_maturity_change,

    MAX(remaining_maturity_change)
        AS max_remaining_maturity_change,

    SUM(
        CASE
            WHEN remaining_maturity_change IS NOT NULL
                 AND NOT isfinite(remaining_maturity_change)
            THEN 1
            ELSE 0
        END
    ) AS nonfinite_maturity_change

FROM read_parquet('{TEMPORAL_FEATURE_FILE}')
"""

temporal_diag_df = feature_con.execute(
    temporal_diag_query
).fetchdf()

print("\nTemporal-feature diagnostics:")
display(temporal_diag_df)

# ------------------------------------------------------------
# 5. NULL behaviour when prior month is invalid
# ------------------------------------------------------------

null_logic_query = f"""
SELECT

    SUM(
        CASE
            WHEN has_valid_prior_month = 0
                 AND prior_month_balance IS NOT NULL
            THEN 1
            ELSE 0
        END
    ) AS invalid_rows_with_balance_lag,

    SUM(
        CASE
            WHEN has_valid_prior_month = 0
                 AND balance_change IS NOT NULL
            THEN 1
            ELSE 0
        END
    ) AS invalid_rows_with_balance_change,

    SUM(
        CASE
            WHEN has_valid_prior_month = 0
                 AND interest_rate_change_1m IS NOT NULL
            THEN 1
            ELSE 0
        END
    ) AS invalid_rows_with_rate_change,

    SUM(
        CASE
            WHEN has_valid_prior_month = 0
                 AND loan_age_change IS NOT NULL
            THEN 1
            ELSE 0
        END
    ) AS invalid_rows_with_age_change,

    SUM(
        CASE
            WHEN has_valid_prior_month = 0
                 AND remaining_maturity_change IS NOT NULL
            THEN 1
            ELSE 0
        END
    ) AS invalid_rows_with_maturity_change

FROM read_parquet('{TEMPORAL_FEATURE_FILE}')
"""

null_logic_df = feature_con.execute(
    null_logic_query
).fetchdf()

print(
    "\nTemporal NULL-logic validation:"
)

display(null_logic_df)

# ------------------------------------------------------------
# 6. Loan-age consistency
# ------------------------------------------------------------

age_consistency_query = f"""
SELECT COUNT(*) AS mismatches
FROM read_parquet('{TEMPORAL_FEATURE_FILE}')
WHERE has_valid_prior_month = 1
  AND loan_age IS NOT NULL
  AND prior_month_loan_age IS NOT NULL
  AND loan_age_change !=
      loan_age - prior_month_loan_age
"""

age_consistency_df = feature_con.execute(
    age_consistency_query
).fetchdf()

age_mismatches = int(
    age_consistency_df.loc[
        0,
        "mismatches"
    ]
)

print(
    "\nLoan-age formula mismatches:",
    age_mismatches
)

# ------------------------------------------------------------
# 7. Final PASS / FAIL checks
# ------------------------------------------------------------

observations_ok = (
    int(basic_df.loc[0, "observations"])
    == 20_155_269
)

loans_ok = (
    int(basic_df.loc[0, "unique_loans"])
    == 450_106
)

events_ok = (
    int(basic_df.loc[0, "events"])
    == 646_957
)

non_events_ok = (
    int(basic_df.loc[0, "non_events"])
    == 19_508_312
)

dates_ok = (
    str(
        basic_df.loc[
            0,
            "min_reporting_date"
        ]
    )[:10]
    == "2018-01-01"
    and
    str(
        basic_df.loc[
            0,
            "max_reporting_date"
        ]
    )[:10]
    == "2025-09-01"
)

null_core_ok = (
    int(
        basic_df.loc[
            0,
            "null_loan_id"
        ]
    ) == 0
    and
    int(
        basic_df.loc[
            0,
            "null_reporting_date"
        ]
    ) == 0
    and
    int(
        basic_df.loc[
            0,
            "null_target"
        ]
    ) == 0
)

columns_ok = (
    len(missing_temporal_columns) == 0
)

duplicates_ok = (
    duplicate_groups == 0
)

prior_flag_ok = (
    int(
        temporal_diag_df.loc[
            0,
            "min_prior_flag"
        ]
    ) >= 0
    and
    int(
        temporal_diag_df.loc[
            0,
            "max_prior_flag"
        ]
    ) <= 1
)

finite_ok = (
    int(
        temporal_diag_df.loc[
            0,
            "nonfinite_balance_change"
        ]
    ) == 0
    and
    int(
        temporal_diag_df.loc[
            0,
            "nonfinite_balance_change_pct"
        ]
    ) == 0
    and
    int(
        temporal_diag_df.loc[
            0,
            "nonfinite_rate_change"
        ]
    ) == 0
    and
    int(
        temporal_diag_df.loc[
            0,
            "nonfinite_age_change"
        ]
    ) == 0
    and
    int(
        temporal_diag_df.loc[
            0,
            "nonfinite_maturity_change"
        ]
    ) == 0
)

null_logic_ok = (
    int(
        null_logic_df.loc[
            0,
            "invalid_rows_with_balance_lag"
        ]
    ) == 0
    and
    int(
        null_logic_df.loc[
            0,
            "invalid_rows_with_balance_change"
        ]
    ) == 0
    and
    int(
        null_logic_df.loc[
            0,
            "invalid_rows_with_rate_change"
        ]
    ) == 0
    and
    int(
        null_logic_df.loc[
            0,
            "invalid_rows_with_age_change"
        ]
    ) == 0
    and
    int(
        null_logic_df.loc[
            0,
            "invalid_rows_with_maturity_change"
        ]
    ) == 0
)

age_formula_ok = (
    age_mismatches == 0
)

# ------------------------------------------------------------
# 8. Final result
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("VALIDATION RESULT")
print("=" * 70)

checks = {
    "Observation count": observations_ok,
    "Unique loan count": loans_ok,
    "Event count": events_ok,
    "Non-event count": non_events_ok,
    "Reporting-date range": dates_ok,
    "No NULL core fields": null_core_ok,
    "Required temporal columns": columns_ok,
    "No duplicate loan-months": duplicates_ok,
    "Prior-month flag valid": prior_flag_ok,
    "Temporal features finite": finite_ok,
    "Invalid-prior NULL logic": null_logic_ok,
    "Loan-age formula": age_formula_ok
}

for name, passed in checks.items():
    print(
        f"{name:<34}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

overall_pass = all(checks.values())

print("\nOVERALL:", "PASS" if overall_pass else "FAIL")
print("=" * 70)

FULL TEMPORAL / DYNAMIC FEATURE DATASET VALIDATION


,observations,unique_loans,events,non_events,reporting_months,min_reporting_date,max_reporting_date,null_loan_id,null_reporting_date,null_target
0,20155269,450106,646957.0,19508312.0,93,2018-01-01,2025-09-01,0.0,0.0,0.0



Total output columns: 66
Required temporal columns checked: 13
Missing temporal columns: 0
All required temporal columns are present.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Duplicate loan-month groups: 0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Temporal-feature diagnostics:


,min_prior_flag,max_prior_flag,valid_prior_month_rows,invalid_prior_month_rows,min_balance_change,max_balance_change,nonfinite_balance_change,min_balance_change_pct,max_balance_change_pct,nonfinite_balance_change_pct,min_interest_rate_change_1m,max_interest_rate_change_1m,nonfinite_rate_change,min_loan_age_change,max_loan_age_change,nonfinite_age_change,min_remaining_maturity_change,max_remaining_maturity_change,nonfinite_maturity_change
0,0,1,19584821.0,570448.0,-661399.3,1299082.16,0.0,-0.987377,9.750541,0.0,-2.875,1.0,0.0,0.0,4.0,0.0,-351.0,347.0,0.0



Temporal NULL-logic validation:


,invalid_rows_with_balance_lag,invalid_rows_with_balance_change,invalid_rows_with_rate_change,invalid_rows_with_age_change,invalid_rows_with_maturity_change
0,0.0,0.0,0.0,0.0,0.0



Loan-age formula mismatches: 0

VALIDATION RESULT
Observation count                 : PASS
Unique loan count                 : PASS
Event count                       : PASS
Non-event count                   : PASS
Reporting-date range              : PASS
No NULL core fields               : PASS
Required temporal columns         : PASS
No duplicate loan-months          : PASS
Prior-month flag valid            : PASS
Temporal features finite          : PASS
Invalid-prior NULL logic          : PASS
Loan-age formula                  : PASS

OVERALL: PASS


## Final Modelling Feature Dataset Assembly

The validated financial, payment-history, and temporal feature layers are
combined into a single modelling-ready dataset.

The temporal-feature dataset already contains all previously engineered
features. Therefore, this step only creates a final consolidated Parquet
file without recalculating or modifying feature values.

The final dataset will retain:

- loan identifier,
- reporting date,
- six-month early-warning target,
- borrower and loan characteristics,
- payment-history behavioural features,
- financial and loan-derived features,
- month-over-month temporal features.

The current delinquency status and explicitly excluded post-event variables
are not reintroduced.

This dataset will be used as the master input for subsequent temporal
train/validation/test splitting and model development.

In [14]:
# ============================================================
# CELL 20 — FINAL MODELLING FEATURE DATASET ASSEMBLY
# ============================================================

FINAL_FEATURE_FILE = (
    FEATURE_OUTPUT_DIR /
    "fannie_2018Q1_final_modelling_features.parquet"
)

print("=" * 70)
print("FINAL MODELLING FEATURE DATASET ASSEMBLY")
print("=" * 70)

print("\nSource:")
print(TEMPORAL_FEATURE_FILE)

print("\nOutput:")
print(FINAL_FEATURE_FILE)

# ------------------------------------------------------------
# Remove previous output if it exists
# ------------------------------------------------------------

if FINAL_FEATURE_FILE.exists():
    FINAL_FEATURE_FILE.unlink()

# ------------------------------------------------------------
# Final dataset creation
# ------------------------------------------------------------

final_feature_query = f"""
COPY (
    SELECT *
    FROM read_parquet(
        '{TEMPORAL_FEATURE_FILE}'
    )
)
TO '{FINAL_FEATURE_FILE}'
(FORMAT PARQUET, COMPRESSION ZSTD);
"""

print("\nCreating final modelling dataset...")
print("Approximately 20.15M observations will be copied.")

feature_con.execute(final_feature_query)

print("\nAssembly completed.")

print(
    "Output exists:",
    FINAL_FEATURE_FILE.exists()
)

if FINAL_FEATURE_FILE.exists():

    size_mb = (
        FINAL_FEATURE_FILE.stat().st_size
        / (1024 ** 2)
    )

    print(
        f"Output size: {size_mb:,.2f} MB"
    )

print("=" * 70)

FINAL MODELLING FEATURE DATASET ASSEMBLY

Source:
C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning\data\processed\feature_engineering\fannie_2018Q1_temporal_features.parquet

Output:
C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning\data\processed\feature_engineering\fannie_2018Q1_final_modelling_features.parquet

Creating final modelling dataset...
Approximately 20.15M observations will be copied.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Assembly completed.
Output exists: True
Output size: 520.23 MB


## Final Master Modelling Dataset — Comprehensive Audit

This is the final quality-control audit of the consolidated modelling
dataset before model development.

The audit verifies:

1. Dataset size and unique-loan count.
2. Event and non-event counts.
3. Reporting-date coverage.
4. Target completeness and binary structure.
5. Loan-month uniqueness.
6. Required modelling columns.
7. Data types.
8. Missingness in engineered features.
9. Infinite and non-finite numeric values.
10. Constant features that provide no modelling information.
11. Presence of excluded leakage-sensitive variables.

The target and all feature values are checked without modifying the
dataset.

A PASS confirms that the final dataset is suitable for the next stage:
temporal train/validation/test splitting and model development.

In [15]:
# ============================================================
# CELL 21 — FINAL MASTER MODELLING DATASET AUDIT
# ============================================================

print("=" * 70)
print("FINAL MASTER MODELLING DATASET — COMPREHENSIVE AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# 1. Basic dataset audit
# ------------------------------------------------------------

basic_query = f"""
SELECT
    COUNT(*) AS observations,
    COUNT(DISTINCT loan_id) AS unique_loans,

    SUM(
        CASE WHEN target_6m = 1
        THEN 1 ELSE 0 END
    ) AS events,

    SUM(
        CASE WHEN target_6m = 0
        THEN 1 ELSE 0 END
    ) AS non_events,

    COUNT(DISTINCT reporting_date)
        AS reporting_months,

    MIN(reporting_date)
        AS min_reporting_date,

    MAX(reporting_date)
        AS max_reporting_date,

    SUM(
        CASE WHEN loan_id IS NULL
        THEN 1 ELSE 0 END
    ) AS null_loan_id,

    SUM(
        CASE WHEN reporting_date IS NULL
        THEN 1 ELSE 0 END
    ) AS null_reporting_date,

    SUM(
        CASE WHEN target_6m IS NULL
        THEN 1 ELSE 0 END
    ) AS null_target

FROM read_parquet('{FINAL_FEATURE_FILE}')
"""

basic_df = feature_con.execute(
    basic_query
).fetchdf()

display(basic_df)

# ------------------------------------------------------------
# 2. Target distribution
# ------------------------------------------------------------

target_query = f"""
SELECT
    target_6m,
    COUNT(*) AS observations,
    ROUND(
        100.0 * COUNT(*) /
        SUM(COUNT(*)) OVER (),
        4
    ) AS percentage
FROM read_parquet('{FINAL_FEATURE_FILE}')
GROUP BY target_6m
ORDER BY target_6m
"""

target_df = feature_con.execute(
    target_query
).fetchdf()

print("\nTarget distribution:")
display(target_df)

# ------------------------------------------------------------
# 3. Schema / required columns
# ------------------------------------------------------------

schema_df = feature_con.execute(
    f"""
    DESCRIBE
    SELECT *
    FROM read_parquet('{FINAL_FEATURE_FILE}')
    """
).fetchdf()

available_columns = schema_df[
    "column_name"
].tolist()

available_column_set = set(
    available_columns
)

required_columns = [
    "loan_id",
    "reporting_date",
    "target_6m",

    # Payment-history features
    "ph_missing",
    "ph_prior_3m_delinquency",
    "ph_prior_6m_delinquency",
    "ph_prior_12m_delinquency",
    "ph_prior_24m_delinquency",
    "ph_prior_max_delinquency",
    "ph_prior_delinquency_count_6m",
    "ph_prior_delinquency_count_12m",
    "ph_prior_delinquency_count_24m",

    # Financial features
    "balance_utilization",
    "interest_rate_change",
    "ltv_cltv_gap",
    "loan_age_ratio",
    "remaining_term_ratio",
    "co_borrower_score_gap",

    "reporting_year",
    "reporting_month",

    # Missingness indicators
    "borrower_credit_score_missing",
    "co_borrower_credit_score_missing",
    "mortgage_insurance_missing",

    # Temporal features
    "prior_month_balance",
    "balance_change",
    "balance_change_pct",
    "prior_month_interest_rate",
    "interest_rate_change_1m",
    "prior_month_loan_age",
    "loan_age_change",
    "prior_month_remaining_maturity",
    "remaining_maturity_change",
    "has_valid_prior_month"
]

missing_required_columns = [
    col
    for col in required_columns
    if col not in available_column_set
]

print("\nSchema audit:")
print("Total output columns:", len(available_columns))
print(
    "Required columns checked:",
    len(required_columns)
)
print(
    "Missing required columns:",
    len(missing_required_columns)
)

if missing_required_columns:
    for col in missing_required_columns:
        print(" -", col)
else:
    print("All required modelling columns are present.")

# ------------------------------------------------------------
# 4. Leakage-sensitive column audit
# ------------------------------------------------------------

excluded_columns = [
    "current_delinquency_status",
    "zero_balance_code",
    "zero_balance_effective_date",
    "mi_cancellation_indicator",
    "interest_only_first_pi_date",
    "months_to_amortization"
]

present_excluded_columns = [
    col
    for col in excluded_columns
    if col in available_column_set
]

print("\nLeakage-sensitive exclusion audit:")

if present_excluded_columns:
    print("WARNING — excluded columns present:")
    for col in present_excluded_columns:
        print(" -", col)
else:
    print(
        "PASS — no explicitly excluded "
        "leakage-sensitive columns are present."
    )

# ------------------------------------------------------------
# 5. Duplicate loan-month audit
# ------------------------------------------------------------

duplicate_query = f"""
SELECT COUNT(*) AS duplicate_groups
FROM (
    SELECT
        loan_id,
        reporting_date,
        COUNT(*) AS n
    FROM read_parquet('{FINAL_FEATURE_FILE}')
    GROUP BY loan_id, reporting_date
    HAVING COUNT(*) > 1
)
"""

duplicate_df = feature_con.execute(
    duplicate_query
).fetchdf()

duplicate_groups = int(
    duplicate_df.loc[0, "duplicate_groups"]
)

print(
    "\nDuplicate loan-month groups:",
    duplicate_groups
)

# ------------------------------------------------------------
# 6. Numeric-column audit
# ------------------------------------------------------------

numeric_types = {
    "DOUBLE",
    "FLOAT",
    "DECIMAL",
    "INTEGER",
    "BIGINT",
    "SMALLINT",
    "TINYINT",
    "HUGEINT"
}

numeric_columns = []

for _, row in schema_df.iterrows():

    dtype = str(
        row["column_type"]
    ).upper()

    if any(
        dtype.startswith(t)
        for t in numeric_types
    ):
        numeric_columns.append(
            row["column_name"]
        )

print(
    "\nNumeric columns detected:",
    len(numeric_columns)
)

# ------------------------------------------------------------
# 7. Non-finite numeric values
# ------------------------------------------------------------

nonfinite_results = []

for col in numeric_columns:

    query = f"""
    SELECT COUNT(*) AS n
    FROM read_parquet('{FINAL_FEATURE_FILE}')
    WHERE "{col}" IS NOT NULL
      AND NOT isfinite("{col}")
    """

    n = int(
        feature_con.execute(query)
        .fetchdf()
        .loc[0, "n"]
    )

    if n > 0:
        nonfinite_results.append(
            (col, n)
        )

print("\nNon-finite numeric values:")

if nonfinite_results:
    for col, n in nonfinite_results:
        print(
            f" - {col}: {n:,}"
        )
else:
    print("PASS — no non-finite numeric values found.")

# ------------------------------------------------------------
# 8. Constant numeric features
# ------------------------------------------------------------

constant_results = []

for col in numeric_columns:

    query = f"""
    SELECT
        COUNT(DISTINCT "{col}") AS distinct_values
    FROM read_parquet('{FINAL_FEATURE_FILE}')
    """

    distinct_values = int(
        feature_con.execute(query)
        .fetchdf()
        .loc[0, "distinct_values"]
    )

    if distinct_values <= 1:
        constant_results.append(
            (col, distinct_values)
        )

print("\nConstant numeric features:")

if constant_results:
    for col, n in constant_results:
        print(
            f" - {col}: {n} distinct value(s)"
        )
else:
    print(
        "PASS — no constant numeric features found."
    )

# ------------------------------------------------------------
# 9. Missingness of engineered features
# ------------------------------------------------------------

engineered_columns = [
    "ph_missing",
    "ph_prior_3m_delinquency",
    "ph_prior_6m_delinquency",
    "ph_prior_12m_delinquency",
    "ph_prior_24m_delinquency",
    "ph_prior_max_delinquency",
    "ph_prior_delinquency_count_6m",
    "ph_prior_delinquency_count_12m",
    "ph_prior_delinquency_count_24m",

    "balance_utilization",
    "interest_rate_change",
    "ltv_cltv_gap",
    "loan_age_ratio",
    "remaining_term_ratio",
    "co_borrower_score_gap",

    "borrower_credit_score_missing",
    "co_borrower_credit_score_missing",
    "mortgage_insurance_missing",

    "prior_month_balance",
    "balance_change",
    "balance_change_pct",
    "prior_month_interest_rate",
    "interest_rate_change_1m",
    "prior_month_loan_age",
    "loan_age_change",
    "prior_month_remaining_maturity",
    "remaining_maturity_change",
    "has_valid_prior_month"
]

missingness_query_parts = []

for col in engineered_columns:

    missingness_query_parts.append(
        f"""
        SUM(
            CASE WHEN "{col}" IS NULL
            THEN 1 ELSE 0 END
        ) AS "{col}"
        """
    )

missingness_query = f"""
SELECT
    {",".join(missingness_query_parts)}
FROM read_parquet('{FINAL_FEATURE_FILE}')
"""

missingness_df = feature_con.execute(
    missingness_query
).fetchdf()

missingness_long = (
    missingness_df
    .T
    .reset_index()
)

missingness_long.columns = [
    "feature",
    "null_count"
]

missingness_long["null_percentage"] = (
    100.0 *
    missingness_long["null_count"]
    /
    int(basic_df.loc[0, "observations"])
)

print("\nEngineered-feature missingness:")
display(
    missingness_long
    .sort_values(
        "null_count",
        ascending=False
    )
    .head(30)
)

# ------------------------------------------------------------
# 10. Final checks
# ------------------------------------------------------------

observations_ok = (
    int(basic_df.loc[0, "observations"])
    == 20_155_269
)

loans_ok = (
    int(basic_df.loc[0, "unique_loans"])
    == 450_106
)

events_ok = (
    int(basic_df.loc[0, "events"])
    == 646_957
)

non_events_ok = (
    int(basic_df.loc[0, "non_events"])
    == 19_508_312
)

reporting_months_ok = (
    int(basic_df.loc[0, "reporting_months"])
    == 93
)

dates_ok = (
    str(
        basic_df.loc[
            0,
            "min_reporting_date"
        ]
    )[:10]
    == "2018-01-01"
    and
    str(
        basic_df.loc[
            0,
            "max_reporting_date"
        ]
    )[:10]
    == "2025-09-01"
)

core_nulls_ok = (
    int(
        basic_df.loc[
            0,
            "null_loan_id"
        ]
    ) == 0
    and
    int(
        basic_df.loc[
            0,
            "null_reporting_date"
        ]
    ) == 0
    and
    int(
        basic_df.loc[
            0,
            "null_target"
        ]
    ) == 0
)

target_binary_ok = (
    set(
        target_df["target_6m"].tolist()
    ).issubset({0, 1})
)

columns_ok = (
    len(missing_required_columns) == 0
)

leakage_ok = (
    len(present_excluded_columns) == 0
)

duplicates_ok = (
    duplicate_groups == 0
)

nonfinite_ok = (
    len(nonfinite_results) == 0
)

constant_ok = (
    len(constant_results) == 0
)

# ------------------------------------------------------------
# 11. Final result
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL MASTER DATASET AUDIT RESULT")
print("=" * 70)

final_checks = {

    "Observation count":
        observations_ok,

    "Unique loan count":
        loans_ok,

    "Event count":
        events_ok,

    "Non-event count":
        non_events_ok,

    "Reporting-month count":
        reporting_months_ok,

    "Reporting-date range":
        dates_ok,

    "No NULL core fields":
        core_nulls_ok,

    "Binary target":
        target_binary_ok,

    "Required modelling columns":
        columns_ok,

    "Excluded columns absent":
        leakage_ok,

    "No duplicate loan-months":
        duplicates_ok,

    "No non-finite numeric values":
        nonfinite_ok,

    "No constant numeric features":
        constant_ok
}

for name, passed in final_checks.items():

    print(
        f"{name:<38}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

overall_pass = all(
    final_checks.values()
)

print("\n" + "=" * 70)
print(
    "OVERALL:",
    "PASS" if overall_pass else "FAIL"
)
print("=" * 70)

FINAL MASTER MODELLING DATASET — COMPREHENSIVE AUDIT


,observations,unique_loans,events,non_events,reporting_months,min_reporting_date,max_reporting_date,null_loan_id,null_reporting_date,null_target
0,20155269,450106,646957.0,19508312.0,93,2018-01-01,2025-09-01,0.0,0.0,0.0



Target distribution:


,target_6m,observations,percentage
0,0,19508312,96.7901
1,1,646957,3.2099



Schema audit:
Total output columns: 66
Required columns checked: 33
Missing required columns: 0
All required modelling columns are present.

Leakage-sensitive exclusion audit:
PASS — no explicitly excluded leakage-sensitive columns are present.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (6 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Engineered-feature missingness:


,feature,null_count,null_percentage
14,co_borrower_score_gap,11397242.0,56.547209
6,ph_prior_delinquency_count_6m,10315042.0,51.177893
7,ph_prior_delinquency_count_12m,10315042.0,51.177893
8,ph_prior_delinquency_count_24m,10315042.0,51.177893
20,balance_change_pct,3210003.0,15.926371
26,remaining_maturity_change,700720.0,3.476610
25,prior_month_remaining_maturity,700648.0,3.476252
21,prior_month_interest_rate,570448.0,2.830267
23,prior_month_loan_age,570448.0,2.830267
18,prior_month_balance,570448.0,2.830267



FINAL MASTER DATASET AUDIT RESULT
Observation count                     : PASS
Unique loan count                     : PASS
Event count                           : PASS
Non-event count                       : PASS
Reporting-month count                 : PASS
Reporting-date range                  : PASS
No NULL core fields                   : PASS
Binary target                         : PASS
Required modelling columns            : PASS
Excluded columns absent               : PASS
No duplicate loan-months              : PASS
No non-finite numeric values          : PASS
No constant numeric features          : PASS

OVERALL: PASS
